# 01/07 贝叶斯定理与统计思维 — 学习 Notebook

**阅读全译本**：[../docs/zh.md](../docs/zh.md)　 **英文原文**：[../docs/en.md](../docs/en.md)

**学习状态：待开始。** 本 Notebook 用于保留你后续独立提交的代码、数学证明、实际测试与导师讲评。正式测验尚未进行，不能提前标记课程完成。

课程主线：**问题 → 概念 → Build It → Use It → Ship It → Exercises → Post Quiz**。官方答案文件 `../quiz.json` 含正确答案，建议正式后测前暂不查看。

参考的可运行官方程序：[bayes.py](bayes.py)。导师不预先填写需要你独立完成的答案。

## 热身：衔接上一课 01/06

上一课你独立推导了条件概率公式，掌握了“两个条件概率方向不能随意互换”。

**思考（先不查答案）**：已知 $P(A\mid B)$ 与 $P(B\mid A)$，什么时候二者会相等？只知道其中一个通常能算出另一个吗？

第一段教学将从 **$P(A\cap B)$ 的两种展开方式**开始，推导贝叶斯定理，而不是直接记忆公式。

## 概念工作区：先验、似然、证据和后验

请在后续课堂中写出两种联合概率展开、证明贝叶斯定理，并用“疾病罕见、99% 灵敏度、1% 假阳性率”的例子说明为什么 $P(\text{sick}\mid\text{positive})$ 并不是 99%。

本节预留作为你的原始推导、疑问和纠错记录。

In [ ]:
# 数值实验工作区：先由你独立实现 bayes(...)，再和官方代码进行对照。
# 不在此处预填练习答案。
import math


## Build It：朴素贝叶斯垃圾邮件分类器

- 在给定类别时，为什么能够把特征的条件似然相乘？
- 未见词的概率为什么可能为零？拉普拉斯平滑怎样处理？
- 为什么要使用 log 概率而不是很多概率直接相乘？
- 实现后再与官方 [bayes.py](bayes.py) 以及 scikit-learn 的 `MultinomialNB` 对照。

**你的独立代码和测试将追加在这里。**

## MLE / MAP、Beta–Binomial 与 A/B 测试

等待依次学习：MLE 与 MAP 的不同、Gaussian 先验与 L2 正则化、Beta 共轭先验、顺序与批次更新的等价关系、A/B 测试中的后验比较。

**官方四道练习**：两次阳性检测、平滑系数变化、加入邮件长度特征、Beta(2,2) 先验下的 MAP 估计。

学生提交后再添加可复现实验和讲评。

## 官方后测与课程总结（待完成）

原版 [quiz.json](../quiz.json) 的后测共有 3 题；必须按顺序逐题回答并给出评分，最后才更新仓库的 `LEARNING.md`。

下一步：读 [../docs/zh.md](../docs/zh.md) 的“问题”“概念”部分，开始 01/07 的教学。

## 第 1 讲：从条件概率到贝叶斯推理——问题与上一课热身

**课程状态：已开始教学，等待学生回答热身，不计分。**

官方 01/07 的核心问题：某疾病患病率为每一万人 1 人；检测的灵敏度为 99%，假阳性率为 1%。某人检测为阳性，实际患病的后验概率是多少？

**两个容易混淆的条件方向：**
- 问题 A：**已经患病**的前提下，得到**阳性结果**的概率；
- 问题 B：**已经得到阳性结果**的前提下，**实际患病**的概率。

直觉上，它们不是同一个问题。一个阳性结果既可能来自真正患病者的真阳性，也可能来自庞大健康人群中少量的假阳性。必须考虑疾病的先验基准率以及两类人的比例，而不能见到“灵敏度 99%”就直接认定“阳性后患病概率 99%”。

**本轮暂停在“问题 → 概念”的直觉阶段**，后续导师将从联合概率的两种分解推导贝叶斯公式，使用 100 万人的自然频数表验证，并逐步完成官方 Build It、Use It、Ship It、四道 Exercises 和 3 道 post quiz。

### 热身问题（继承 01/06，等待学生作答，不计分）

1. 单标签交叉熵 $L=-\ln p_y$：对同一真实类别预测概率为 $0.9$ 与 $0.01$ 时，哪种预测损失更大？为什么？
2. 语言模型的 token 概率连乘很小：为什么改在对数域求和？得到的总 log probability 在数值上有限，是否意味着用 $\exp$ 恢复原始概率永远不会下溢？

**指导原则：** 在学生回答前不提前给出答案，不替学生填写手写代码。收到回答后保存学生原话、数学讲评、可运行例子和下一道题。

## 第 1 讲续：热身讲评（2/2，非正式测验）

学生原话：“问题1 0.01的损失更大,因为负对数似然是单调递减的,因为log probalilities可以把连乘转换成加法2,不能保证,”

### 第一题：One-hot 分类交叉熵

**正确。** 令 $L(p_y)=-\ln p_y$，有

$$\frac{dL}{dp_y}=-\frac{1}{p_y}<0\quad(0<p_y\le1).$$

因此 $p_y=0.01$ 时损失 $4.605170\ldots$，高于 $p_y=0.9$ 时的 $0.105360\ldots$。

术语澄清：**$\ln p$ 严格递增，$-\ln p$ 严格递减**，学生指的是负对数损失，判断无误。

### 第二题：为什么对数域计算不代表恢复原始概率永远不会下溢？

**正确。** $\ln\prod_i p_i=\sum_i\ln p_i$，在 log 域求和可以避免很多极小概率在相乘阶段下溢。但在总 log probability 很小时，最终调用 $\exp(\sum_i\ln p_i)$，仍然可能发生浮点下溢为 $0.0$。数学上的正概率与浮点零必须区分。

这两题是不计分的上一课知识复习，不影响 01/07 未来官方 post quiz 的评分。

## 第 2 讲：从联合概率严格推导 Bayes' Theorem

### 先区分“条件方向”

假设 $D$ 表示**患病**，$+$ 表示**检测阳性**。

- $P(+\mid D)$：在患病者中，检测出现阳性的概率，即**灵敏度**。
- $P(D\mid +)$：在阳性者中，实际上患病的概率，即**阳性预测值/后验概率**。

两者不可以直接交换，因为条件概率的分母不同：

$$P(D\mid +)=\frac{P(D\cap +)}{P(+)},\qquad
P(+\mid D)=\frac{P(D\cap +)}{P(D)}.$$

两个式子都包含同一个联合概率 $P(D\cap +)$，因此

$$P(D\mid +)P(+)=P(+\mid D)P(D).$$

整理：

$$\boxed{P(D\mid +)=\frac{P(+\mid D)P(D)}{P(+)}}.$$

对于“检测阳性”，考虑全部可能来源——**患者中的真阳性**与**健康者中的假阳性**。全概率公式给出

$$P(+)=P(+\mid D)P(D)+P(+\mid\neg D)P(\neg D).$$

于是二元假设下的贝叶斯公式：

$$\boxed{P(D\mid +)=
\frac{P(+\mid D)P(D)}
{P(+\mid D)P(D)+P(+\mid\neg D)(1-P(D))}}.$$

- $P(D)$：先验 Prior，即患病率；
- $P(+\mid D)$：似然 Likelihood，即灵敏度；
- $P(+)$：证据 Evidence，即总体阳性概率；作为归一化分母；
- $P(D\mid +)$：后验 Posterior，观察阳性后的患病概率。

### 100 万人的自然频数实验（不使用抽样误差）

课文条件：$P(D)=0.0001$，$P(+\mid D)=0.99$，$P(+\mid\neg D)=0.01$。

想象恰好统计 **1,000,000 人**，并用上述比例计算期望人数：

| 人群 | 全部人数 | 检测阳性 | 检测阴性 |
|---|---:|---:|---:|
| 患病 | 100 | 99（真阳性） | 1 |
| 健康 | 999,900 | 9,999（假阳性） | 989,901 |
| 合计 | 1,000,000 | **10,098** | 989,902 |

在所有检测阳性的 $10,098$ 人中，只有 $99$ 人真正患病，因此

$$P(D\mid +)=\frac{99}{10098}=0.0098039215686\ldots
\approx \boxed{0.9804\%}.$$

用概率公式计算完全一致：

$$P(+)=0.99\cdot0.0001+0.01\cdot0.9999=0.010098,$$

$$P(D\mid +)=\frac{0.99\cdot0.0001}{0.010098}
\approx0.0098039215686.$$

**重要假设**：“检测 99% 准确”并不单独决定后验；上面的算例明确假定患者灵敏度为 99%，健康者假阳性率为 1%。用于临床推断还需真实患病率与实际检测性能等信息。

### 导师独立运行的数字验证（非学生手写 Bayes 函数）

本单元检验概率法与自然频数法数值一致。实际由导师 Python 运行得到：`evidence=0.010098`、`posterior=0.009803921569`，即 `0.980392%`。

In [ ]:
import math

prior = 0.0001
sensitivity = 0.99
false_positive_rate = 0.01

evidence = sensitivity * prior + false_positive_rate * (1 - prior)
posterior = sensitivity * prior / evidence

population = 1_000_000
sick = 100
healthy = 999_900
true_positives = 99
false_positives = 9_999

assert math.isclose(evidence,0.010098,rel_tol=1e-12)
assert math.isclose(posterior,99/10098,rel_tol=1e-12)
assert true_positives + false_positives == 10_098
assert sick + healthy == population

print(f"Evidence P(positive): {evidence:.10f}")
print(f"Posterior P(sick|positive): {posterior:.12f}")
print(f"Positive predictive value: {100*posterior:.6f}%")

## 第 2 讲理解检查（等待学生回答）

当患病率保持 $P(D)=0.0001$、灵敏度保持 $P(+\mid D)=0.99$ 不变，仅把健康者的**假阳性率从 1% 降至 0.1%**（$0.01\rightarrow0.001$）时：

1. 得到阳性结果后，真正患病的概率会**上升还是下降**？请用贝叶斯公式解释分母会怎样变化。
2. 尝试算出新的 $P(D\mid +)$，可以保留到百分比小数点后两位。

学生回答后，导师再验收并进入官方 Build It 第 1 步：由学生独立实现 `bayes(prior, likelihood, false_positive_rate)`。这里暂时**不提前填写完整函数答案**。

## 第 2 讲理解检查：学生解答与讲评（2/2）

**学生原始思路**：“问题1 检测到阳性,真正患病 是p(D|+) = [贝叶斯二元公式] 分子不变, 分母减少,整体增加,患病概率会升高, 问题2 是0.09”。

**判定：两题正确。** 第一问的推理严谨，明确看出了真阳性概率（分子）不变，健康人假阳性概率降低使整体阳性概率（分母）下降，所以检测阳性后的实际患病后验上升。第二问写为 `0.09`，应理解为**小数概率 0.09 = 9%**，与正确答案相符的近似值；若误写成 `0.09%` 则有 100 倍单位误差。

设患病率 $p=0.0001$、灵敏度 $s=0.99$、假阳性率 $f=0.001$，则

$$P(D\mid+)=\frac{sp}{sp+f(1-p)}
=\frac{0.000099}{0.000099+0.0009999}
=\frac{0.000099}{0.0010989}
\approx 0.09009009009.$$ 

以百分比表示为：

$$\boxed{P(D\mid+)\approx 9.01\%}.$$

同样的先验与灵敏度，若 $f=0.01$，原后验为 $0.00980392157\approx0.98\%$。由此可见降低假阳性率十倍，会让阳性预测值显著升高，但**并不是严格增加十倍**：因为分母中的真阳性项 $sp$ 是固定的。

### 用自然频数验证（1000 万人）

- 患病约 1000 人，其中检测阳性 $990$ 人；
- 健康约 $9,999,000$ 人，其中假阳性 $9,999$ 人；
- 所有阳性共 $990+9999=10,989$ 人，真正患病者占 $990/10,989\approx0.09009009$。

这里的数字是对一个理想化 1000 万人总体按理论比例计算的期望人数，真实抽样有波动。

**必须区分**：`0.09` 表示约 $9\%$，而 `0.09%` 表示小数概率 $0.0009$。在机器学习中混写小数概率与百分比是一种常见且影响严重的工程错误。

### 导师独立执行的数值验证（3/3 通过）

以下代码独立测试原始 1% 与改进 0.1% 假阳性率两种场景，并验证数值与自然频数方法一致。由导师 Python 环境运行通过。

In [ ]:
import math

prevalence = 0.0001
sensitivity = 0.99

def posterior_reference(prior, likelihood, false_positive_rate):
    numerator = prior * likelihood
    evidence = numerator + false_positive_rate * (1 - prior)
    return numerator / evidence

old_posterior = posterior_reference(prevalence, sensitivity, 0.01)
new_posterior = posterior_reference(prevalence, sensitivity, 0.001)

checks = {
    "original natural frequencies": math.isclose(old_posterior, 99/10098, rel_tol=1e-12),
    "improved natural frequencies": math.isclose(new_posterior, 990/10989, rel_tol=1e-12),
    "improved posterior increases": new_posterior > old_posterior
}
assert all(checks.values()), checks

print("Old false positive 1%:", old_posterior, "=", f"{old_posterior:.2%}")
print("New false positive 0.1%:", new_posterior, "=", f"{new_posterior:.2%}")
print(f"PASS {sum(checks.values())}/{len(checks)}")

## 第 3 讲：官方 Build It 第一步——亲手实现贝叶斯定理函数

已经理解公式、分母来源与假阳性率的作用，下一步由**学生独立写 Python**，而不是只看原作者实现：

```python
def bayes(prior, likelihood, false_positive_rate):
    # prior: P(D)，患病率
    # likelihood: P(+ | D)，检测灵敏度
    # false_positive_rate: P(+ | not D)，健康人假阳性率
    # 返回 P(D | +)
    pass
```

使用刚推导的完整二元公式

$$P(D\mid+)=\frac{P(+\mid D)\,P(D)}{P(+\mid D)\,P(D)+P(+\mid\neg D)(1-P(D))}.$$

学生提交后导师独立验证：
- 原版：`bayes(0.0001,0.99,0.01)` 约为 `0.0098039215686`；
- 改进版：`bayes(0.0001,0.99,0.001)` 约为 `0.09009009009`；
- 明确的特殊情况：`prior=1` 应得到 1（当检测灵敏度非零）；`prior=0` 应得到 0（当假阳性率非零）；
- 若证据概率分母为零，说明该阳性事件在模型假设下不可能出现，条件概率没有定义。可抛出 `ValueError`；不要把“后验=0”当作数学事实。

目前处于 Build It 阶段；尚未进入 01/07 正式 post quiz，不更新 `LEARNING.md` 的完成状态。

## 第 3 讲（验收）：学生的 Bayes 实现、边界辨析

### 学生独立提交的原始代码

学生正确写出了一个分子（真阳性的联合概率）除以完整证据概率（真阳性 + 假阳性）的计算流程。这一版本对应官方教程 Build It 第 1 步，且**不调用任何外部机器学习库**。

学生对于零分母的解释：“意味着没有这个检查到一个有症状的, 不能返回0”。

**评价**：“不能返回 0”正确；更严谨的说法是：分母 $P(+)=0$，表示按照当前概率模型，**检测阳性事件本身概率为零**，并不是“没有症状”（$+$ 代表检测阳性，不代表症状）。在基本条件概率定义 $P(D\mid+)=P(D\cap+)/P(+)$ 中，若 $P(+)=0$，条件概率**未定义**。

学生原代码在零分母时会自动抛 `ZeroDivisionError`，符合这一数学事实。为了给业务 API 提供清晰异常消息，可在明确需要时增加 `if evidence == 0: raise ValueError(...)`，但这属于导师推荐的增强，不替换学生正确的原始实现。

另外，先验为 0 并不一定导致零分母：若 $P(D)=0$ 且假阳性率 $P(+\mid\neg D)>0$，总体仍会出现假阳性，后验合法地等于 0。先验为 1、灵敏度非零时，后验合法地等于 1。

**导师已实际独立执行 13/13 项测试**，涵盖 1% 与 0.1% 假阳性率、先验边界、完美检测器、仅健康者出现阳性、后验范围、随着先验和假阳性率变化的单调性，以及三组零证据概率的 `ZeroDivisionError`。

### 学生原始实现（完整保留，不替换为导师修改版）

In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    # prior: P(D)
    # likelihood: P(+ | D)
    # false_positive_rate: P(+ | not D)
    # 返回 P(D | +)
    return (likelihood * prior) / (likelihood * prior + false_positive_rate * (1 - prior))

### 导师实际运行的测试：13/13（可复现）

In [ ]:
import math

bayes_test_results = {
    "official case": math.isclose(bayes(.0001,.99,.01),.000099/.010098,rel_tol=1e-12),
    "lower false positive rate": math.isclose(bayes(.0001,.99,.001),.000099/.0010989,rel_tol=1e-12),
    "zero prior with positive false positive": bayes(0,.99,.01)==0,
    "certain prior with positive likelihood": bayes(1,.99,.01)==1,
    "equal likelihood and FPR keeps prior": math.isclose(bayes(.32,.7,.7),.32,rel_tol=1e-12),
    "perfect detector": bayes(.14,1.,0.)==1.,
    "only healthy gives positive": bayes(.14,0.,.25)==0.,
    "posterior within [0,1]": all(0<=bayes(p,s,f)<=1 for p,s,f in [
        (.2,.4,.6),(.9,.6,.15),(.02,.99,.01)]),
    "lower FPR gives higher posterior": bayes(.0001,.99,.001)>bayes(.0001,.99,.01),
    "higher prior gives higher posterior": bayes(.5,.6,.1)>bayes(.1,.6,.1),
}

for i, args in enumerate([(0.,.99,0.),(1.,0.,.01),(.5,0.,0.)],start=1):
    try:
        bayes(*args)
    except ZeroDivisionError:
        bayes_test_results[f"zero evidence raises {i}"] = True
    else:
        bayes_test_results[f"zero evidence raises {i}"] = False

assert len(bayes_test_results)==13
assert all(bayes_test_results.values()), bayes_test_results
print(f"Bayes student's original implementation: {sum(bayes_test_results.values())}/13 passed")
print("Original FPR:",bayes(.0001,.99,.01))
print("Reduced FPR:",bayes(.0001,.99,.001))

## 第 4 讲：朴素贝叶斯（Naive Bayes），从“一个证据”扩展到“一封邮件里的许多词”

### 真实问题

有一批已标注为 spam（垃圾邮件）或 ham（正常邮件）的训练消息。对于一条新消息，例如 “free meeting”，我们想知道它应该属于哪个类别。

贝叶斯公式适用于“类别 $C$”与“邮件词特征 $w_1,\ldots,w_n$”：

$$P(C\mid w_1,\ldots,w_n)
=\frac{P(w_1,\ldots,w_n\mid C)P(C)}{P(w_1,\ldots,w_n)}.$$

难点在 $P(w_1,\ldots,w_n\mid C)$：直接统计每一种词序列组合，训练样本远远不够。

### 朴素性 = 类别已知时的条件独立假设

朴素贝叶斯假定 **给定类别 $C$ 后**，词特征按所选模型可视为条件独立，因此：

$$P(w_1,\ldots,w_n\mid C)\approx\prod_i P(w_i\mid C).$$

这不等于说“在所有邮件中，单词天然相互独立”，也不等于说“邮件类别和单词独立”。例如 “New York” 这种短语通常不独立，故称“朴素”假设。

对候选类别而言，分母 $P(w_1,\ldots,w_n)$ 相同，做分类 **argmax** 时可忽略：

$$\hat C=\arg\max_C\left[P(C)\prod_iP(w_i\mid C)\right].$$

数值实现改为对数域求和：

$$\boxed{\hat C=\arg\max_C\left[\log P(C)+\sum_i\log P(w_i\mid C)\right]}.$$

这直接复用了第 06 课学生自己学过的“对数把连乘变成加法”这一技巧。

### 训练时究竟要统计哪些数？

官方教程使用 **Multinomial Naive Bayes** 风格的 token 词频模型：

- `class_counts[C]`：某类别中有多少**封训练邮件**，用于估计 $P(C)$；
- `word_counts[C][w]`：该类别的训练邮件里单词 $w$ **总共出现多少次**，一个邮件重复出现两次就计两次；
- `class_word_totals[C]`：该类别所有训练邮件中的 token 总数；
- `vocab`：整个训练集出现过哪些不同单词。

估计类别先验：

$$P(C)=\frac{\mathrm{class\_counts}[C]}{\mathrm{total\_documents}}.$$

为什么还需要“拉普拉斯平滑”？若某词在某类训练数据里出现 0 次，直接用词频给它分配 0 概率，则对数会变成 $-\infty$。采用加 $\alpha$ 的平滑规则（$\alpha>0$）：

$$\boxed{P(w\mid C)=\frac{\mathrm{word\_counts}[C][w]+\alpha}
{\mathrm{class\_word\_totals}[C]+\alpha|\mathrm{vocab}|}}.$$

注意分母是**该类别全部 token 数**加上平滑质量，而不是该类别邮件的封数；这与简单统计“某词出现在多少封邮件里”属于不同特征模型。

### 先练训练阶段，不急着实现全部分类器

**任务**：你自己写 `NaiveBayes` 的初始化与 `train` 两个方法，实现文档数、词频、总词数、词表的统计。仅需 Python 标准库。

```python
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # self.smoothing
        # self.class_counts: 每个类的邮件数
        # self.word_counts: 每个类每个词的总出现次数
        # self.class_word_totals: 每个类的总 token 数
        # self.vocab: 训练集全部不同的词
        pass

    def train(self, documents, labels):
        # 对每一封邮件及其标签进行遍历
        # doc.lower().split() 得到词列表
        # 每处理一个 token，更新相应词频、类别总 token 数和词表
        pass
```

**小样本验证**：

```python
model = NaiveBayes()
model.train(
    ["free free win", "meeting schedule"],
    ["spam", "ham"]
)
assert model.class_counts["spam"] == 1
assert model.class_counts["ham"] == 1
assert model.word_counts["spam"]["free"] == 2
assert model.class_word_totals["spam"] == 3
assert model.class_word_totals["ham"] == 2
assert model.vocab == {"free", "win", "meeting", "schedule"}
```

**思考题**：在这个例子中，单词 “free” 只出现在第一封邮件，但累计出现 2 次。为什么朴素贝叶斯的词频模型会记录 `2` 而不是 `1`？“邮件中出现过这个词”与“这个词出现了多少次”分别对应什么样的特征？

收到你写的两个方法后，再实施真实测试并进入 `predict`、拉普拉斯平滑和概率对照。

## 第 4 讲：NaiveBayes 训练统计——学生代码验收

**学生原始回答**：“因为统计的是每一个单词出现的所有次数,如果不统计某个单词出现的总次数,这样我们就丢失了他们的总体的出现的比例,只会知道这个单词出现过”。

解释正确：多项式朴素贝叶斯记录 token 出现次数；伯努利特征只记录某个词是否在文档中出现。对于 `free free win`，多项式词频 `free=2`，而文档是否出现该词为 1。在这两封训练邮件组成的词表 $|V|=4$ 下，以 $\alpha=1$ 做加一平滑：

$$P(\mathrm{free}|\mathrm{spam})=(2+1)/(3+4)=3/7,\quad
P(\mathrm{free}|\mathrm{ham})=(0+1)/(2+4)=1/6.$$

**导师真实执行结果**：基础 7 项检查 **7/7 通过**：类别文档数 2 项、重复词次数、spam/ham token 总数、词表、smoothing 属性。

**与官方代码的预处理行为比较**：**0/3 通过**。官方 `code/bayes.py` 使用 `doc.lower().split()`；学生写的是 `words.split(' ')`，有以下区别：

1. `"Free free WIN"`：学生将三个词按大小写区分，官方统一小写后 `free=2`。
2. `"free  win"`：学生多统计了空字符串 `""`，错误地把 token 总数算成 3 而不是 2。
3. `"meeting\tschedule"`：学生未分开制表符，误将整段计为一个 token。

**本题状态：基础统计正确，文本规范化待修订。** 暂不提前给出改好的整个类，也不替学生修改代码。学生应独立在现有 `train` 中统一 `lower()` 并改用不带参数的 `split()`。

附加工程提醒：`zip(documents, labels)` 遇长度不等会静默截断，生产级输入校验可检查长度一致；这是额外建议，不是本练习的失败项。

### 学生原始代码（未经导师替换）

In [ ]:
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.split(' '):
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

### 可复现测试和学生修订要求

以下代码检查学生的原始版本，基础检查通过；官方预处理一致性检查**预计失败**，按字典逐项打印，避免未修订前整个 Notebook 在 `assert` 上中断。学生修订后可复用相同测试验证 3 项是否全部通过。

**只需修订分词逻辑**，不必重写已经正确的四类统计结构。修好后把完整的 `train` 方法发给导师进行重新验收，再实现 `predict`。

In [ ]:
base=NaiveBayes()
base.train(["free free win","meeting schedule"],["spam","ham"])
basic={
    "spam documents": base.class_counts["spam"]==1,
    "ham documents": base.class_counts["ham"]==1,
    "repeated free token": base.word_counts["spam"]["free"]==2,
    "spam total tokens": base.class_word_totals["spam"]==3,
    "ham total tokens": base.class_word_totals["ham"]==2,
    "vocabulary": base.vocab=={"free","win","meeting","schedule"},
    "smoothing stored": base.smoothing==1.0,
}
assert all(basic.values())
case=NaiveBayes()
case.train(["Free free WIN"],["spam"])
spaces=NaiveBayes()
spaces.train(["free  win"],["spam"])
tabs=NaiveBayes()
tabs.train(["meeting\tschedule"],["ham"])
edge={
    "lowercase": case.vocab=={"free","win"} and case.word_counts["spam"]["free"]==2,
    "consecutive spaces": spaces.vocab=={"free","win"} and spaces.class_word_totals["spam"]==2,
    "tab separator": tabs.vocab=={"meeting","schedule"} and tabs.class_word_totals["ham"]==2,
}
print("Basic:",sum(basic.values()),"/",len(basic))
print("Official tokenization:",sum(edge.values()),"/",len(edge))
print("Failing cases:",[name for name,passed in edge.items() if not passed])

## 第 4 讲修订验收：学生 NaiveBayes 初始化与训练统计通过（13/13）

学生独立修改了 `train()` 的分词表达式，从 `words.split(' ')` 改为 `words.lower().split()`。**符合官方 `code/bayes.py` 的预处理行为。**

导师在实际 Python 环境调用学生提交的完整代码，执行 **13 项测试，13/13 通过**：

- 原先的 7 项基础统计：spam、ham 文档数量；单词重复次数；spam、ham token 总数；训练词汇表；smoothing 属性。
- 原先不通过的 3 项官方分词一致性检查：统一大小写、连续空格、制表符分词。**现在 3/3 通过。**
- 另外补充的 3 项检查：含换行与混合空白的 token 分词、首尾空白不会生成空词、非默认 smoothing 参数正常保存。**3/3 通过。**

本次修订只修复预处理，保留学生自己选择的数据结构、原始注释和统计方式。

### 学生提出的新问题：“使用多项式模型及拉普拉斯平滑时这个是干什么的？”

**多项式朴素贝叶斯 Multinomial Naive Bayes** 关心某类别的训练材料里，单词共出现多少个 token，因此 `free free win` 中 `free` 的计数是 2 而不是 1。它使用类别条件 token 词频近似词分布，而不仅仅是判断“是否出现过这个词”。伯努利朴素贝叶斯的词存在特征则将该词视为 0/1。

**拉普拉斯平滑 Laplace Smoothing** 关心有限训练数据中从未见过的词。没有平滑时，某类训练中词计数为 0，估计的条件词概率就为 0，整个词概率的乘积可能被一个未见词归零；使用 log probability 时则出现 $\log 0$ 的问题。

对多项式 token 词频模型的有限训练词汇表 $V$：

$$P(w\mid C)=\frac{\mathrm{count}(w,C)+\alpha}{N_C+\alpha|V|},\qquad\alpha>0.$$

这里 $N_C$ 是类别 $C$ 中全部 token 的计数，$|V|$ 是**全训练集不同词的数量**。分母需要额外增加 $\alpha|V|$，因为词汇表里的每个词都获得了 $\alpha$ 伪计数，整个分布必须归一化；`train()` 阶段只存计数，不应在训练中反复修改计数来“添加平滑”。

### 具体计算

训练数据：

```python
documents = ["free free win", "meeting schedule"]
labels = ["spam", "ham"]
```

有 $N_{\mathrm{spam}}=3,\ N_{\mathrm{ham}}=2,\ |V|=4$。对于 $\alpha=1$：

| 单词 | spam 原始次数 | $P(w\mid\mathrm{spam})$ | ham 原始次数 | $P(w\mid\mathrm{ham})$ |
|---|---:|---:|---:|---:|
| free | 2 | $3/7$ | 0 | $1/6$ |
| win | 1 | $2/7$ | 0 | $1/6$ |
| meeting | 0 | $1/7$ | 1 | $2/6$ |
| schedule | 0 | $1/7$ | 1 | $2/6$ |

每个类别的四种词概率之和都是 1。$P(\mathrm{free}\mid\mathrm{spam})=3/7$ 与 $P(\mathrm{free}\mid\mathrm{ham})=1/6$ 的区别提供分类证据；这是后续 `predict()` 的基础。

对 `free meeting`，无平滑时两个类别的原始词似然乘积均为零；平滑后每个词在训练词汇表的每个类别里获得非零概率，因此可以有效比较。

**$\alpha$ 越大**，伪计数的影响越强，各词概率趋向更均匀；$\alpha$ 越接近 0，估计越接近直接使用原始词频。对于测试时不在训练词汇表内的 OOV 词，需要另行约定“忽略或映射至 UNK”等规则；直接套用训练词汇表的分母去给任意多个新词分配概率，不会构成归一化的词分布。这是后续工程扩展，不影响此处已知词的学习。

### 学生修订后的完整原始代码（严格保留）

In [ ]:
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

### 导师实际运行的 13 项测试（修订后全通过）

In [ ]:
train_base=NaiveBayes()
train_base.train(["free free win","meeting schedule"],["spam","ham"])
train_checks={
    "spam doc count":train_base.class_counts["spam"]==1,
    "ham doc count":train_base.class_counts["ham"]==1,
    "repeated free token":train_base.word_counts["spam"]["free"]==2,
    "spam token total":train_base.class_word_totals["spam"]==3,
    "ham token total":train_base.class_word_totals["ham"]==2,
    "vocab":train_base.vocab=={"free","win","meeting","schedule"},
    "default smoothing":train_base.smoothing==1.,
}
train_case=NaiveBayes()
train_case.train(["Free free WIN"],["spam"])
train_spaces=NaiveBayes()
train_spaces.train(["free  win"],["spam"])
train_tabs=NaiveBayes()
train_tabs.train(["meeting\tschedule"],["ham"])
train_checks.update({
    "lowercase":train_case.vocab=={"free","win"} and train_case.word_counts["spam"]["free"]==2,
    "repeated spaces":train_spaces.vocab=={"free","win"} and train_spaces.class_word_totals["spam"]==2,
    "tabs":train_tabs.vocab=={"meeting","schedule"} and train_tabs.class_word_totals["ham"]==2,
})
train_extra=NaiveBayes(.5)
train_extra.train(["  FREE \n free\tWIN \n","meeting\t schedule"],["spam","ham"])
train_checks.update({
    "newlines":train_extra.word_counts["spam"]["free"]==2 and train_extra.word_counts["spam"]["win"]==1,
    "trim blanks":"" not in train_extra.vocab and train_extra.class_word_totals["spam"]==3,
    "custom smoothing attr":train_extra.smoothing==.5,
})
assert len(train_checks)==13 and all(train_checks.values()),train_checks
print(f"Student NaiveBayes training: {sum(train_checks.values())}/13 passed.")

## 下一步：检验自己对拉普拉斯平滑的理解，然后实现词的 log likelihood

训练阶段已通过。此时 `self.smoothing` **只是被保存的参数**，还没有作用于任何概率计算。

**先回答两个问题：**
1. 给定 `free free win`（spam）和 `meeting schedule`（ham），为什么用拉普拉斯平滑时分母需要加 $\alpha|V|$，而不只是加 $\alpha$？
2. 已知 $\alpha=1$、$|V|=4$，你能根据原始词频亲手算出 $P(\mathrm{meeting}\mid\mathrm{spam})$ 和 $P(\mathrm{meeting}\mid\mathrm{ham})$ 吗？

若已掌握，请进一步**独立实现** `_log_likelihood(self,word,cls)`，返回 $\ln P(word\mid cls)$，并从 `self.word_counts`、`self.class_word_totals`、`len(self.vocab)` 和 `self.smoothing` 读取数据。后续将使用同一方法编写 `predict()`。

## 拉普拉斯平滑：学生理解核验

**学生原话**：“我们要保证整体的概率和为1”。

**判定：正确。** 对每个固定类别 C，需要保证训练词汇表 V 上单词条件概率之和为 1：

$$\sum_{w\in V} P(w\mid C)=1.$$

每个词增加一个平滑伪计数 alpha，整个词表的计数总共增加 alpha × |V|：

$$\sum_{w\in V}[\operatorname{count}(w,C)+\alpha]=N_C+\alpha|V|.$$

举例：训练数据 free free win 属于 spam；词表包括 free、win、meeting、schedule。原始计数分别 2、1、0、0，alpha=1 后变成 3、2、1、1，总和由 3 增为 7；分母若是 7，概率和为 1；分母若错误地写成 4，概率和为 7/4，违反归一化。

**学习进度：** 学生已理解多项式词频统计与拉普拉斯平滑的必要性；下一步计算具体条件词概率，再编写 log likelihood；尚未完成课后测验。

## 练习：计算不同类别的平滑条件概率（等待作答）

训练两封邮件：spam 类别的内容是 free free win；ham 类别的内容是 meeting schedule。

完整训练词汇表由 free、win、meeting、schedule 四个词组成，即 |V|=4。采用 alpha=1。

请分别计算：

1. $P(\mathrm{meeting}\mid\mathrm{spam})$；
2. $P(\mathrm{meeting}\mid\mathrm{ham})$。

写出每一项的分子和分母，并解释为何两个类别的分母可能不同。

此题通过后由学生独立实现 _log_likelihood(self, word, cls)，使用学习过的平滑公式和自然对数。

## 拉普拉斯平滑手算：学生答案核验（2/2）

学生原始回答：

> P(meeting|spam) = 1 / (3 + 4) = 1 / 7; P(meeting|ham) = (1 + 1) / ( 2 + 4) = 1/3

**评分：2/2，完全正确**。已正确掌握“分子加平滑系数，分母加整个训练词表的平滑总计数”。

固定训练集：

- spam: free free win，token 总数为 3，meeting 原始计数为 0。
- ham: meeting schedule，token 总数为 2，meeting 原始计数为 1。
- 整体训练词汇表包含 4 个词，alpha=1。

所以：

$$P(\mathrm{meeting}\mid\mathrm{spam})=\frac{0+1}{3+1\times 4}=\frac17\approx0.142857;$$

$$P(\mathrm{meeting}\mid\mathrm{ham})=\frac{1+1}{2+1\times 4}=\frac13\approx0.333333.$$

两个类别的分母不同，是因为它们的原始 token 总数分别是 3 和 2；平滑所加的词表伪计数总数都是 4。该词在 ham 条件下更常见，因此是支持 ham 的证据，但**绝不能把似然 $P(\mathrm{meeting}\mid\mathrm{ham})$ 误写成后验 $P(\mathrm{ham}\mid\mathrm{meeting})$**。

接下来的独立任务是编写一个使用这些平滑概率的 log-likelihood 方法；不提前替学生写出方法体。

## Build It — 下一步：独立实现 _log_likelihood()

给现有 NaiveBayes 类添加一个方法，返回指定词在给定类别下的**对数条件概率**。

方法签名：

```python
def _log_likelihood(self, word, cls):
    # 返回 log P(word | cls)
    # 用已有的 self.word_counts、self.class_word_totals、
    # self.smoothing、self.vocab 计算
    pass
```

所需公式：

$$P(w\mid C)=\frac{\mathrm{count}(w,C)+\alpha}{N_C+\alpha|V|}.$$

$$\log P(w\mid C)=\ln\left(\frac{\mathrm{count}(w,C)+\alpha}{N_C+\alpha|V|}\right).$$

提示：Python 标准库 `math.log(x)` 是自然对数。请先实现已在训练词表中的词，不急着处理训练时未见的 OOV 词和空训练数据。

学生实现后，用以下值验证：

```python
model = NaiveBayes(smoothing=1.0)
model.train(["free free win", "meeting schedule"], ["spam", "ham"])
assert math.isclose(model._log_likelihood("meeting", "spam"), math.log(1/7))
assert math.isclose(model._log_likelihood("meeting", "ham"), math.log(1/3))
assert math.isclose(model._log_likelihood("free", "spam"), math.log(3/7))
```

下一步再推导类别先验 `P(C)`、总的 log-class score 和 `predict()`。

In [ ]:
import math
assert math.isclose((0 + 1)/(3 + 1*4), 1/7)
assert math.isclose((1 + 1)/(2 + 1*4), 1/3)
print('Conditional probabilities verified: 2/2')

## Build It — 学生 log likelihood 初稿验收：3/9

学生原始方法：

    def _log_likelihood(self, word, cls):
            # 返回 log P(word | cls)
            return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + len(self.vocab))

**真实执行结果：默认 alpha=1 时 3/3 通过；alpha=0.5、2 时合计 0/6 通过。**

学生正确应用对数变换：$\log(a/b)=\log a-\log b$。问题是，分母里漏了 alpha 倍的词表大小。正式平滑公式：

$$P(w\mid C)=\frac{\operatorname{count}(w,C)+\alpha}{N_C+\alpha|V|}.$$

原实现分母实际使用 $N_C+|V|$。只有 alpha=1 时，它才等于 $N_C+\alpha|V|$。

| alpha | 词/类别 | 原始输出 | 正确结果 | 测试 |
|---|---|---:|---:|---|
| 1 | meeting/spam | -1.945910149 | -1.945910149 | 通过 |
| 1 | meeting/ham | -1.098612289 | -1.098612289 | 通过 |
| 1 | free/spam | -0.847297860 | -0.847297860 | 通过 |
| 0.5 | meeting/spam | -2.639057330 | -2.302585093 | 未通过 |
| 0.5 | meeting/ham | -1.386294361 | -0.980829253 | 未通过 |
| 0.5 | free/spam | -1.029619417 | -0.693147181 | 未通过 |
| 2 | meeting/spam | -1.252762968 | -1.704748092 | 未通过 |
| 2 | meeting/ham | -0.693147181 | -1.203972804 | 未通过 |
| 2 | free/spam | -0.559615788 | -1.011600912 | 未通过 |

此前学生已经理解每个固定类别下词概率应当和为 1。这里恰好能验证：spam 原始 token 总数是 3、训练词表有 4 个词；alpha=0.5 时应加 2 个伪计数，正确分母是 5，原始函数用 7，导致概率总和变成 5/7；alpha=2 时应加 8 个伪计数，分母为 11，原始函数仍用 7，使概率总和变成 11/7。

**本阶段暂不标记全部通过。** 等学生独立补齐分母中的 alpha 系数，再用同一组跨平滑系数测试复核。原代码在下一单元原样保留。

In [ ]:
from collections import defaultdict
import math

# 以前学生通过验收的 train() 代码 + 本轮学生的原始 _log_likelihood() 方法
class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        # 返回 log P(word | cls)
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + len(self.vocab))

In [ ]:
import math
results=[]
for alpha in (1.,.5,2.):
    m=NaiveBayes(smoothing=alpha)
    m.train(["free free win","meeting schedule"],["spam","ham"])
    for word,cls in (("meeting","spam"),("meeting","ham"),("free","spam")):
        actual=m._log_likelihood(word,cls)
        expected=math.log((m.word_counts[cls][word]+alpha)/(m.class_word_totals[cls]+alpha*len(m.vocab)))
        ok=math.isclose(actual,expected,rel_tol=1e-12,abs_tol=1e-12)
        results.append(ok)
        print(f"alpha={alpha:g}, {word}/{cls}, actual={actual:.9f}, expected={expected:.9f}: {'PASS' if ok else 'FAIL'}")
assert len(results)==9
print(f"Student first attempt: {sum(results)}/{len(results)} checks passed (expected 3/9 before repair).")

## 下一步：学生独立修正通用平滑系数

只需调整原方法的**分母**，确保它和数学公式一致：$N_C+\alpha|V|$。不要改变正确的对数相减结构、学生的训练实现和原始计数。

用 alpha=1、0.5、2 三种平滑系数重新运行前述 9 项测试，目标是 **9/9 全部通过**。还可以手工交叉检查：

- 当 alpha=0.5，spam 的 meeting 词概率应为 $0.5/(3+0.5\times4)=0.1$，log probability 应为 $\ln(0.1)\approx-2.302585$。
- 当 alpha=2，spam 的 meeting 词概率应为 $2/(3+2\times4)=2/11$，log probability 为 $\ln(2/11)\approx-1.704748$。

**重要**：此单元保留首稿和首稿测试结果，修订版应单独追加；不直接覆盖学生原始历史。完成后再学习类别先验与 predict。

## Build It · 学生第二次提交 log-likelihood：修订通过

学生原始方法在本单元后续代码块中完整保留。

**结论：修正完全正确。** 多项式朴素贝叶斯加 alpha 平滑的对数概率为：

$$\log P(w\mid C)=
\log(\mathrm{count}(w,C)+\alpha)-
\log(N_C+\alpha|V|).$$

修订后的代码分子、分母均与公式一致，且正确使用了 $\ln(a/b)=\ln a-\ln b$。上一次错误仅在分母漏乘了平滑系数 alpha。

导师实际执行 **15/15 项检查**：alpha 为 0.5、1、2 时，对三个词与类别组合比较数值（9 项）；每个 alpha 下分别验证 spam 与 ham 的概率分布归一化（6 项）。全部通过。

| alpha | log P(meeting given spam) | 预期值 | 结果 |
|---|---:|---:|---|
| 0.5 | -2.302585093 | -2.302585093 | PASS |
| 1 | -1.945910149 | -1.945910149 | PASS |
| 2 | -1.704748092 | -1.704748092 | PASS |

**测试边界**：只验证训练词表中的词、非空训练集与正平滑参数；未涵盖未知词 OOV、空训练集、非法 alpha。因此当前通过不意味着生产级边界已完整处理。

**状态**：已完成 train() 与 log likelihood；下一步学习类别先验 log prior，再实现 predict()。正式课后测验尚未开始。

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        # 返回 log P(word | cls)
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

In [ ]:
import math
results = []
for alpha in (0.5, 1.0, 2.0):
    m = NaiveBayes(smoothing=alpha)
    m.train(["free free win", "meeting schedule"], ["spam", "ham"])
    for word, cls in (("meeting", "spam"), ("meeting", "ham"), ("free", "spam")):
        actual = m._log_likelihood(word, cls)
        expected = math.log((m.word_counts[cls][word]+alpha)/(m.class_word_totals[cls]+alpha*len(m.vocab)))
        passed = math.isclose(actual, expected, rel_tol=1e-12, abs_tol=1e-12)
        results.append(passed)
        print(f"alpha={alpha}, {word}/{cls}, observed={actual:.9f}, expected={expected:.9f}: {'PASS' if passed else 'FAIL'}")
    for cls in ("spam", "ham"):
        total_prob = sum(math.exp(m._log_likelihood(w, cls)) for w in m.vocab)
        passed = math.isclose(total_prob, 1., rel_tol=1e-12, abs_tol=1e-12)
        results.append(passed)
        print(f"alpha={alpha}, {cls} normalization={total_prob:.12f}: {'PASS' if passed else 'FAIL'}")
assert len(results) == 15 and all(results), results
print(f"Revised log likelihood: {sum(results)}/15 passed.")

## 下一题：独立实现类别先验 _log_prior()

你已经写出了训练阶段的计数结构。类别先验不是某个词的出现次数，而是各类别占所有训练邮件的比例：

$$P(C)=\frac{\mathrm{class\_counts}[C]}{\sum_{C'}\mathrm{class\_counts}[C']}.$$

$$\log P(C)=\ln(P(C)).$$

现在给 NaiveBayes 添加：

    def _log_prior(self, cls):
        # 返回 log P(cls)
        pass

请使用已有的 self.class_counts 和 Python 的 math.log，完成方法，不必调用机器学习库。

**测试数据：**

    model = NaiveBayes()
    model.train(["free free win", "meeting schedule", "project meeting", "schedule review"],
                ["spam", "ham", "ham", "ham"])

预期：

    model._log_prior("spam") == math.log(1/4)  # 近似比较
    model._log_prior("ham") == math.log(3/4)   # 近似比较

请用 math.isclose 检验浮点结果。

**思考题**：即使某个词同时出现于两类邮件，为什么类别数量不同（先验不同）仍会影响最终的分类结果？

本题通过后，进入 predict()，将 log prior 与逐词 log likelihood 相加后选择得分最大的类别。

## Build It：学生 _log_prior() 验收（9/9）

学生原始代码：

    def _log_prior(self, cls):
            # 返回 log P(cls)
            return math.log(self.class_counts[cls])  - math.log(sum( value for _,value in self.class_counts.items()))

学生的解释：

> 因为就算她出现在两个类别中,但是她出现在两个类别中的次数的比例不一样,可能这个词在5个spam中有,再6个ham中有

**代码正确：9/9 项真实测试通过。** 当前类先验取训练文档中的类别频率，与 token 无关：

$$P(C)=\frac{\mathrm{class\_counts}[C]}{\sum_{C'}\mathrm{class\_counts}[C']},$$
$$\log P(C)=\log\mathrm{class\_counts}[C]-\log \sum_{C'}\mathrm{class\_counts}[C'].$$

四封训练邮件、一封 spam、三封 ham 的案例中，spam 先验为 0.25、log prior 为 -1.3862943611；ham 先验为 0.75、log prior 为 -0.2876820725。

**思考题的解释要区分三件事**：

1. 类别先验 P(C) = 指定类别训练邮件数量 / 所有训练邮件数量，与具体词 w 无关。
2. 条件词似然 P(w|C) = 在指定类别下这个词出现的概率；当前 Multinomial Naive Bayes 使用该类别 token 频率（加 alpha 平滑）。
3. 类别后验 P(C|w) = 观察词之后，根据先验与似然推断类别概率。

学生举出的“这个词在 5 封 spam、6 封 ham 出现”实际是词的**文档出现频率**，不是类别先验，也不等同于当前多项式模型的 token 次数。

**反例**：训练数据有 10 封 spam 和 90 封 ham，且同一个词在两个类别下的条件似然都是 0.2。那么 P(spam|w) = 0.1、P(ham|w) = 0.9，说明当该证据对类别同等可能时，类别先验仍决定后验比例。

**验收项目共 9 个**：两类 log prior、两类概率之和为 1、两类训练文档数量、改动平滑系数保持类别先验不变、同样比例复制训练数据后先验不变、词内容改变但类别比例不变时先验不变。均已执行并通过。

**边界**：学生使用 defaultdict(int)，对未训练过的类别查询会创建零值并在 math.log(0) 处发生 ValueError。本轮只对确实存在于训练集的类别评分，未来可加强空训练集和非法类别的检查。

下一阶段：独立实现 predict()，在 log 域累加类先验与各词似然。

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        # 返回 log P(word | cls)
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

    def _log_prior(self, cls):
        # 返回 log P(cls)
        return math.log(self.class_counts[cls])  - math.log(sum( value for _,value in self.class_counts.items()))

In [ ]:
import math

m=NaiveBayes()
m.train(
    ["free free win", "meeting schedule", "project meeting", "schedule review"],
    ["spam","ham","ham","ham"]
)
checks={
    "spam prior":math.isclose(m._log_prior("spam"),math.log(.25),rel_tol=1e-12),
    "ham prior":math.isclose(m._log_prior("ham"),math.log(.75),rel_tol=1e-12),
    "prior normalization":math.isclose(sum(math.exp(m._log_prior(c)) for c in m.class_counts),1.,rel_tol=1e-12),
    "spam document count":m.class_counts["spam"]==1,
    "ham document count":m.class_counts["ham"]==3,
    "prior expected":math.isclose(m._log_prior("spam"),math.log(.25)),
}
m_half=NaiveBayes(.5)
m_half.train(["free free win", "meeting schedule", "project meeting", "schedule review"], ["spam","ham","ham","ham"])
checks["prior independent of smoothing"]=math.isclose(m_half._log_prior("spam"),m._log_prior("spam"))
m_repeated=NaiveBayes()
m_repeated.train(["free free win"]*2+["meeting schedule"]*6,["spam"]*2+["ham"]*6)
checks["repeating dataset retains class fraction"]=math.isclose(m_repeated._log_prior("ham"),m._log_prior("ham"))
m_changed_words=NaiveBayes()
m_changed_words.train(["meeting meeting meeting","meeting","meeting","meeting"],["spam","ham","ham","ham"])
checks["changing words doesn't change class prior"]=(math.isclose(m_changed_words._log_prior("ham"),m._log_prior("ham")) and math.isclose(m_changed_words._log_prior("spam"),m._log_prior("spam")))
assert len(checks)==9 and all(checks.values()),checks
print(f"Student _log_prior: {sum(checks.values())}/{len(checks)} passed.")
print("spam:",m._log_prior("spam"))
print("ham:",m._log_prior("ham"))

## Build It：独立实现 predict()

现有的训练统计、词对数似然和类对数先验都已验收。现在将它们组合为：

$$\operatorname{score}(C)=\log P(C)+\sum_{i=1}^{n}\log P(w_i\mid C).$$

$$\hat C=\arg\max_C\operatorname{score}(C).$$

请在学生自己的类里新增：

    def predict(self, document):
        # 1. 用与 train 相同的小写及空白分词处理 document
        # 2. 遍历已训练的类别（self.class_counts）
        # 3. 对每个类别从 self._log_prior(cls) 开始
        # 4. 对每个词累加 self._log_likelihood(word, cls)
        # 5. 记录分数最高的类别，返回其名称
        pass

可用 best_score=float("-inf")、best_class=None 记录最大值。

**先只测试训练词汇表里存在的词**，暂不引入 OOV 策略：

    model = NaiveBayes()
    model.train(["free free win","meeting schedule","project meeting","schedule review"],["spam","ham","ham","ham"])
    assert model.predict("free win") == "spam"
    assert model.predict("meeting schedule") == "ham"

思考：ham 类别先验更高，但某封邮件包含非常支持 spam 的词时，为什么仍可能预测为 spam？

学生提交 predict 以后再进行代码审查与测试；官方 post quiz 在完成全课后进行。

## Build It · 学生独立完成 predict()：验收 18/18 通过

学生原始代码在下面的可执行类定义中保留，核心做法：

1. 将邮件规范化为小写并使用默认 split() 按空白分词，与训练预处理一致；
2. 遍历 self.class_counts 中训练过的所有类别；
3. 计算 _log_prior(cls) 加上该邮件所有 token 的 _log_likelihood(word, cls) 之和；
4. 用 max(pre, key=pre.get) 返回分数最高的类别名，而不是分数。

这正好是多项式朴素贝叶斯分类器的对数打分公式：

$$s(C;d)=\log P(C)+\sum_{i=1}^n\log P(w_i\mid C),\qquad \hat C=\arg\max_Cs(C;d).$$

**实际验证：18/18 通过。** 包含 13 个不同邮件输入场景（大小写、重复空格与换行、单词或词组、空消息）和 5 个数学不变量/数字一致性检查。

### 学生的概念解释与纠正

学生原话：“因为它的likelihood只有当cls=spam才有值”。

**这句不正确**：加一平滑让词表中未在某类别出现过的词也拥有正的条件概率。分类结果为 spam 不是因为 ham 类的单词似然为零，而是 spam 的词证据优势超过了 ham 类的先验优势。

本次四封训练邮件里词表大小 |V|=6，spam 有 3 个 token，ham 有 6 个 token；邮件总数为 spam 1、ham 3。

| 概率 | spam | ham |
|---|---:|---:|
| P(C) | 1/4 | 3/4 |
| P(free | C) | 3/9 | 1/12 |
| P(win | C) | 2/9 | 1/12 |

对 free win 的**未归一化**概率得分：

$$S(\mathrm{spam})=\frac14\frac39\frac29=\frac1{54},$$

$$S(\mathrm{ham})=\frac34\frac1{12}\frac1{12}=\frac1{192}.$$

于是 log score 分别约 -3.98898404656 与 -5.25749537203，预测 spam。若只在这两个已训练的类别间归一化，此消息 spam 的模型后验约为 0.780487804878（对应 32/41）。

**关键反例**：对只有一个词 win 的消息：

$$S(\mathrm{spam})=\frac14\frac29=\frac1{18},\quad S(\mathrm{ham})=\frac34\frac1{12}=\frac1{16}.$$

尽管 win 在训练时仅出现于 spam 类别，因为加一平滑且 ham 先验更大，最终 predict("win") 反而返回 ham。这是基准率与似然共同作用的具体证据。

**功能边界**：此处测试限于训练词表内的词。当前 _log_likelihood 对任何未登录词也使用固定词表分母赋予正概率，若将无限多的新词都这样处理，词分布不再归一化。正式对照 scikit-learn 时先选择全部词均出现在训练词表中的消息，OOV 词处理策略另行明确。空训练集没有定义预测类别；本次仅验证正常训练后的空消息回退到类别先验。

### 与 scikit-learn 的小样本实测对照

在本地 Python scikit-learn 1.8.0 环境中，用 CountVectorizer + MultinomialNB(alpha=1.0) 训练同样四封邮件，对 free win、meeting schedule、FREE WIN、free、win、free meeting 六条只含训练词表内词的消息，预测结果分别为 spam、ham、spam、spam、ham、ham，与学生的 predict() 一致。注意 CountVectorizer 默认 token_pattern 等预处理细节可能与学生的 split() 不同，因此这个匹配只证实该测试语料的一致性。

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        # 返回 log P(word | cls)
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

    def _log_prior(self, cls):
        # 返回 log P(cls)
        return math.log(self.class_counts[cls])  - math.log(sum( value for _,value in self.class_counts.items()))

    def predict(self, document):
        # 将邮件转换为小写并分词
        # 遍历 self.class_counts 中的每个类别
        # 初始分数为 self._log_prior(cls)
        # 对每个词累加 self._log_likelihood(word, cls)
        # 返回得分最高的类别
        words = document.lower().split()
        pre = {}
        for cls in self.class_counts:
            pre[cls] = self._log_prior(cls) + sum(self._log_likelihood(word,cls) for word in words)
        return max(pre,key=pre.get)

In [ ]:
import math

m=NaiveBayes()
m.train(
    ["free free win","meeting schedule","project meeting","schedule review"],
    ["spam","ham","ham","ham"]
)
expected={
    "free win":"spam",
    "meeting schedule":"ham",
    "FREE WIN":"spam",
    "MEETING SCHEDULE":"ham",
    "project meeting":"ham",
    "free":"spam",
    "win":"ham",
    "meeting":"ham",
    "schedule":"ham",
    "free meeting":"ham",
    "":"ham",
    "  FREE  \n WIN\t":"spam",
    "free free win":"spam",
}
checks={f"predict {d!r}":m.predict(d)==cls for d,cls in expected.items()}
score=lambda cls,doc: m._log_prior(cls)+sum(m._log_likelihood(w,cls) for w in doc.lower().split())
checks.update({
    "free win spam logscore":math.isclose(score("spam","free win"),math.log(1/54),rel_tol=1e-12),
    "free win ham logscore":math.isclose(score("ham","free win"),math.log(1/192),rel_tol=1e-12),
    "ham free smoothed probability positive":math.isclose(math.exp(m._log_likelihood("free","ham")),1/12,rel_tol=1e-12),
    "ham win smoothed probability positive":math.isclose(math.exp(m._log_likelihood("win","ham")),1/12,rel_tol=1e-12),
    "class prior normalization":math.isclose(sum(math.exp(m._log_prior(c)) for c in m.class_counts),1.,rel_tol=1e-12),
})
assert len(checks)==18 and all(checks.values()),checks
print(f"Student predict: {sum(checks.values())}/{len(checks)} passed")
print("free win scores:",score("spam","free win"),score("ham","free win"))
print("win prediction:",m.predict("win"))

## Use It：用 scikit-learn 的多项式朴素贝叶斯做一致性对照

完成了零依赖的从零实现。进入官方教程 Use It 阶段。

请自己运行以下第三方库对照工作流：

1. 使用 sklearn.feature_extraction.text.CountVectorizer 将训练邮件转换成词频矩阵；
2. 使用 sklearn.naive_bayes.MultinomialNB(alpha=1.0) 进行训练；
3. 对相同训练集和下列六条消息，比较自己实现与 sklearn 的预测：free win、meeting schedule、FREE WIN、free、win、free meeting；
4. 检查每条消息的预测标签是否全部一致，并说说为什么单词 win 单独输入时会被分成 ham。

从下面的接口开始即可，不必使用 GPU：

    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.naive_bayes import MultinomialNB

    docs=["free free win","meeting schedule","project meeting","schedule review"]
    y=["spam","ham","ham","ham"]
    test=["free win","meeting schedule","FREE WIN","free","win","free meeting"]

你独立补全向量化、fit、predict 与结果对比。注意 sklearn 默认分词规则与自写的 lower().split() 并不完全一致，因此暂时只使用训练词表中的英文词做公平对照；之后再讨论 OOV 以及预处理一致性的问题。

本阶段尚未进行 Ship It、共轭先验、MLE/MAP、练习与正式后测，因此不更新 LEARNING.md 为“完成”。

## Use It：学生提出 Scikit-learn 库用法的问题

学生提交了与原教程相同的 CountVectorizer 和 MultinomialNB 示例，询问两个库的作用、基本接口和与自己手写 NaiveBayes 的关系。

**职责对照**

- CountVectorizer：小写化、分词、构建词表与文档词频矩阵。fit_transform 学习训练词表并编码训练数据；transform 利用固定词表编码测试数据。
- MultinomialNB：利用训练类别计算类先验和条件词概率，在 log 域预测。默认平滑系数 alpha=1.0，对应之前学生实现的 self.smoothing=1.0。
- fit(X_train, train_labels) 学习模型统计参数；predict(X_test) 给出标签；predict_proba(X_test) 给出各类别的模型估计概率。
- clf.classes_ 表示 predict_proba 输出每一列的类别；class_log_prior_ 表示各类别对数先验；feature_log_prob_ 表示各类别各特征对数概率。

**为什么不能在测试集上 fit_transform**：重复 fit 会重新确定词表及各列含义，导致训练与推理特征错位，还可能形成错误的数据处理流程。推理应使用先前 fit 好的 vectorizer.transform。

**区别**：CountVectorizer 默认将文本转换为小写，并使用匹配两个及以上单词字符的正则表达式进行 token 化；和学生自己实现的 lower().split() 不完全一样。该例中的单词均至少两个字符且没有标点干扰，因此两种统计规则一致。

**实际独立运行证据**：在 Python 3 环境安装的 scikit-learn 1.8.0 中运行下方可执行单元，训练四封邮件，特征顺序 free、meeting、project、review、schedule、win，得到 4×6 词频矩阵；训练后 class order 是 ham、spam，class log prior 约 -0.28768207、-1.38629436。六条测试消息按顺序预测 spam、ham、spam、spam、ham、ham，匹配学生的手写分类器；free win 的 ham/spam 后验约 0.2195122 / 0.7804878。

**参考官方文档**：
- https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
- https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html

**状态**：讲解库接口并提供可直接执行的代码；学生提供了示例但没有贴自己的运行输出，所以不能把本步骤标为已独立完成。后续继续官方 Ship It 与统计部分。

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

train_docs = [
    'free free win',
    'meeting schedule',
    'project meeting',
    'schedule review'
]
train_labels = ['spam', 'ham', 'ham', 'ham']
test_messages = ['free win', 'meeting schedule', 'FREE WIN', 'free', 'win', 'free meeting']

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_docs)
print('Vocabulary:', vectorizer.get_feature_names_out())
print('Train count matrix:', X_train.toarray())

clf = MultinomialNB(alpha=1.0)
clf.fit(X_train, train_labels)
X_test = vectorizer.transform(test_messages)
predictions = clf.predict(X_test)
probabilities = clf.predict_proba(X_test)

print('Class order:', clf.classes_)
print('Class log prior:', clf.class_log_prior_)
for msg, pred, probs in zip(test_messages, predictions, probabilities):
    print(f'{msg!r} -> {pred}, probabilities={probs}')

## 第 5 讲：MLE、MAP、Beta 共轭先验

学生同意进入课程后续 MLE/MAP 与共轭先验部分。本节从参数估计出发，解释先前实现的多项式朴素贝叶斯及加一平滑。

### MLE

设硬币正面概率为 $\theta$。10 次独立抛掷观察 7 次正面、3 次反面，忽略与参数无关的常数：

$$L(\theta)\propto\theta^7(1-\theta)^3.$$

对数似然：

$$\ell(\theta)=7\log\theta+3\log(1-\theta),\quad
\ell'(\theta)=\frac7\theta-\frac3{1-\theta}.$$

令导数为 0，得到最大似然估计：

$$\boxed{\hat\theta_{\rm MLE}=7/10=0.7.}$$

二阶导数 $-7/\theta^2-3/(1-\theta)^2<0$，验证是内部极大值。

在学生实现的多项式朴素贝叶斯中，条件词概率的 MLE 是**单词 token 次数除以该类别 token 总数**。课程原文某处用“多少封邮件含该词 / 该类邮件封数”描述词概率，这对应词存在的 Bernoulli 文档特征，并非学生正在使用的多项式 token 频率，应予区分。

### MAP

MAP 估计通过最大化参数的后验密度：

$$\hat\theta_{\rm MAP}=\arg\max_\theta p(D\mid\theta)p(\theta).$$

选 Beta$(a,b)$ 参数先验，密度正比于 $\theta^{a-1}(1-\theta)^{b-1}$。采用 Beta$(2,2)$、观察 7 正 3 反：

$$p(\theta\mid D)\propto\theta^{8}(1-\theta)^4,$$

所以后验是 Beta$(9,5)$，后验众数是：

$$\boxed{\hat\theta_{\rm MAP}=\frac{9-1}{9+5-2}=\frac23\approx0.666667.}$$

与 MLE 0.7 不同。高斯参数先验可导出 L2 正则化，但 MAP 与 L2 并非普遍等价。

### 共轭与后验预测

Beta 是 Bernoulli 的共轭先验：

$$\mathrm{Beta}(a,b)+\{s\text{正},f\text{反}\}
\longrightarrow\mathrm{Beta}(a+s,b+f).$$

对于当前后验 Beta$(9,5)$：

- MAP 众数是 $8/12\approx0.6667$；
- 后验均值是 $9/14\approx0.642857$；
- 下一次正面的后验预测概率也是 $9/14$，因为

$$P(H_{\rm next}\mid D)=\int_0^1\theta p(\theta\mid D)d\theta
=\mathbb E[\theta\mid D].$$

MAP 与后验预测不能混淆：一个是后验最密集处的参数点，一个是对全部参数不确定性积分后的预测量。

### 加一平滑的贝叶斯解释

对于 $K=|V|$ 个训练词汇的多项式词频，设词概率向量有对称 Dirichlet$(\alpha,\ldots,\alpha)$ 先验。观测词频 $n_1,\ldots,n_K$ 后，后验为 Dirichlet$(n_1+\alpha,\ldots,n_K+\alpha)$。

下一次词为第 $i$ 个词的后验预测概率：

$$\boxed{P(w_i\mid D)=\frac{n_i+\alpha}{N+K\alpha}.}$$

这就是学生已经实现的 add-alpha 平滑。特别地，alpha=1 时是 Dirichlet$(1,\ldots,1)$ 先验下的后验预测均值；它不是该先验下的 MAP。若换成 Dirichlet$(2,\ldots,2)$ 的后验众数，在适用条件下可以得到相同的加一表达式。

后续将练习参数的序贯更新，并完成正式 Ship It、A/B 测试和本课原始练习。

In [ ]:
from math import isclose
s, f, a, b = 7, 3, 2, 2
mle = s / (s+f)
post_a, post_b = a+s, b+f
map_est = (post_a-1)/(post_a+post_b-2)
pred_est = post_a/(post_a+post_b)
assert isclose(mle, 0.7)
assert (post_a, post_b) == (9,5)
assert isclose(map_est, 2/3)
assert isclose(pred_est, 9/14)
counts=[2,1,0,0]
alpha=1.
predictive=[(n+alpha)/(sum(counts)+alpha*len(counts)) for n in counts]
assert all(isclose(x,y) for x,y in zip(predictive,[3/7,2/7,1/7,1/7]))
assert isclose(sum(predictive),1.)
print('MLE:',mle,'MAP:',map_est,'next head:',pred_est)
print('Dirichlet posterior predictive:',predictive)
print('All checks passed')

## 第 5 讲理解检查（等待学生回答）

先验 Beta$(2,2)$，观察 7 正面、3 反面，参数后验为 Beta$(9,5)$。

**问题 1：** 如果预测下一次抛掷正面，应该用后验众数 $2/3$，还是后验均值 $9/14$？为什么？

**问题 2：** 如果再观察到 3 正面、1 反面，当前后验 Beta$(9,5)$ 变成什么 Beta 分布？请写出两个参数并解释更新方式。

回答后继续序贯与批量更新等价性的证明，再进入 A/B 测试和课程其他练习。

## 第 5 讲理解题：学生解答与点评（2/2）

**学生问题 1 原话**：

“是后验均值,因为通过

$$P(H_{\text{next}}\mid D)=\int_0^1P(H_{\text{next}}\mid\theta)\,p(\theta\mid D)\,d\theta
=\int_0^1\theta\,p(\theta\mid D)\,d\theta
=\mathbb E[\theta\mid D]=\frac9{14}.$$”

**学生问题 2 原话**：“beta(12,6)”

**验收：两题全部正确。**

问题 1 的推导正确。后验预测边缘化未知参数，因此对于下一次独立的伯努利正面事件，预测概率等于 Beta 参数后验均值 $9/14\approx0.642857$，而不是后验众数（MAP）$2/3$。

问题 2 的回答正确。在此前 Beta$(9,5)$ 后验基础上增加 3 次正面、1 次反面，两参数按计数分别相加：

$$\mathrm{Beta}(9,5)\xrightarrow{3H,1T}\boxed{\mathrm{Beta}(12,6)}.$$

### 序贯更新和批量更新为什么等价？

假设同一个固定参数 $\theta$ 描述所有抛硬币实验，且给定 $\theta$ 后每次抛掷条件独立。令第一批充分统计量为 $(s_1,f_1)$、第二批为 $(s_2,f_2)$。

从 Beta$(a,b)$ 出发，序贯更新：

$$\mathrm{Beta}(a,b)\to\mathrm{Beta}(a+s_1,b+f_1)
\to\mathrm{Beta}(a+s_1+s_2,b+f_1+f_2).$$

若直接把两批数据合并，批量更新：

$$\mathrm{Beta}(a,b)\to\mathrm{Beta}(a+s_1+s_2,b+f_1+f_2).$$

两者严格一致。在本例：

- 序贯：Beta$(2,2)$ 加 7H3T 得 Beta$(9,5)$，再加 3H1T 得 Beta$(12,6)$。
- 批量：Beta$(2,2)$ 直接加 10H4T，也得 Beta$(12,6)$。

证明基础是似然在条件独立假设下可乘法分解：

$$p(D_1,D_2\mid\theta)=p(D_1\mid\theta)p(D_2\mid\theta).$$

因此，更新无需保留所有历史原始样本，仅需保留硬币的成功/失败充分统计量。

**适用前提**：同一个 $\theta$、先验一致、观测条件独立、观测规则相同。如硬币偏置随时间变化，固定 $\theta$ 模型就可能不合适，不能把该结论直接推广为所有在线学习算法的性质。

导师独立 Python 实测：先验 Beta$(2,2)$，第一批后验 (9,5)，第二批 (12,6)，批量 (12,6)；后验预测由 $9/14$ 更新为 $12/18=2/3$；所有断言通过。

In [ ]:
from math import isclose
prior=(2,2)
first=(7,3)
second=(3,1)
after_first=(prior[0]+first[0],prior[1]+first[1])
sequential=(after_first[0]+second[0],after_first[1]+second[1])
batch=(prior[0]+first[0]+second[0],prior[1]+first[1]+second[1])
assert after_first == (9,5)
assert sequential == (12,6)
assert batch == (12,6)
assert sequential == batch
assert isclose(after_first[0]/sum(after_first),9/14)
assert isclose(sequential[0]/sum(sequential),2/3)
print('Sequential:',after_first,'=>',sequential)
print('Batch:',batch)
print('Posterior predictive:',after_first[0]/sum(after_first),'=>',sequential[0]/sum(sequential))
print('ALL CHECKS PASSED')

## 第 6 讲：A/B 测试中的贝叶斯更新（下一步）

将 Beta–Bernoulli 更新用于按钮点击率试验。

两组独立实验都使用 Beta$(1,1)$ 先验，各获得 1000 次曝光：
- A：50 次点击，950 次未点击；
- B：65 次点击，935 次未点击。

对每组设点击率为其未知 Bernoulli 参数（$\theta_A$、$\theta_B$），相互独立建模，各有自己的 Beta 后验。

**练习 1**：分别写出 A、B 两组的后验 Beta 参数。

**练习 2**：分别计算两组后验均值（也就是下一次曝光的后验预测点击概率）。

**思考题**：即使 B 的后验均值更大，是否就能说“B 组真实点击率大于 A 组的概率等于 100%”？请解释。

之后将使用 Monte Carlo 抽样估计 $P(\theta_B>\theta_A\mid D)$，并讨论阈值、代价和不确定性。

## A/B Testing：学生回答与数学讲评

学生原始回答：

> 1,beta(51,951) beta(66,936), 2.A是51/1002 b是66/1002,B的点击预测概率更高, 不会,因为随着参数的更新,B的后验可能比A低

**题 1 和题 2：2/2 正确。** 两组都从独立 Beta(1,1) 先验出发：A 的 50 次点击、950 次未点击得到 Beta(51,951)；B 的 65 次点击、935 次未点击得到 Beta(66,936)。

后验预测下一次点击的概率等于后验均值，A = 51/1002 ≈ 0.05089820（5.09%），B = 66/1002 ≈ 0.06586826（6.59%）。

**思考题结论正确，但理由须精确化**：学生提到未来增加数据后排名可能改变，这当然可能，但当前比较概率并非因为未来数据会改变，而是因为在**固定已有数据**时，两个未知点击率的后验分布仍存在不确定性、仍会重叠。即使不再观测，当前也有可能 theta_A > theta_B。

在假设 A 与 B 后验独立时，定义胜出概率：

$$P(\theta_B>\theta_A\mid D)=\int_0^1 f_B(b)F_A(b)\,db.$$

其中 f_B 是 Beta(66,936) 后验密度，F_A 是 Beta(51,951) 后验 CDF。

实际执行数值验证：NumPy 2.3.5，随机种子 20261103，1,000,000 次独立后验抽样，得到 Monte Carlo 比例 0.924316，MC 估计标准误约 0.00026449；SciPy 数值积分得到 0.9244803789025741，积分误差估计 3.5e-13。两者差别属于普通 Monte Carlo 随机误差。

因此 B 优于 A 的当前后验概率约 **92.45%**，不是 100%。如果事先以 95% 后验胜出概率为上线阈值，那么这个概率尚未达到要求；但业务最优决策还应考虑收益与代价，不能只看阈值。

**适用假设**：各变体访问/点击为适当的独立伯努利观测；先验为指定的独立 Beta(1,1)；当前数据可信且没有失配/选择偏差。

至此完成 A/B 场景的共轭更新和后验胜出概率解释，开始官方 Exercises 第 1 道。

In [ ]:
import numpy as np
rng = np.random.default_rng(20261103)
N = 1_000_000
a_samples = rng.beta(51, 951, size=N)
b_samples = rng.beta(66, 936, size=N)
p_better_mc = float(np.mean(b_samples > a_samples))
mc_standard_error = (p_better_mc * (1 - p_better_mc) / N) ** .5
assert abs(p_better_mc - .9244803789025741) < .003
print('P(B>A) MC:', p_better_mc)
print('Monte Carlo SE:', mc_standard_error)
print('Posterior mean A:', 51/1002)
print('Posterior mean B:', 66/1002)
print('Assuming a 0.95 threshold, criterion met?', p_better_mc > .95)

In [ ]:
from scipy.integrate import quad
from scipy.stats import beta
p_exact, integration_error = quad(
    lambda b: beta.pdf(b, 66, 936) * beta.cdf(b, 51, 951),
    0., 1., epsabs=1e-12, epsrel=1e-12
)
assert abs(p_exact - .9244803789025741) < 1e-10
print('P(B>A) numerical integration:', p_exact)
print('Quadrature estimated numerical error:', integration_error)

## 官方 Exercises · 第 1 题：两次独立阳性检测（等待学生）

沿用本课最初的罕见疾病例子：

- 患病率 P(D)=0.0001；
- 灵敏度 P(+|D)=0.99；
- 健康人假阳性率 P(+|not D)=0.01。

现在对同一个人做两次检测，并且假定**在给定其是否患病的条件下**，两次检测结果条件独立。两次都是阳性。

**请用两种方法算出 P(D|+,+)：**

1. 序贯更新：先计算第一次阳性后的后验，并将它作为第二次更新的先验。
2. 一次性更新：利用联合似然 P(+,+|D)=0.99^2 和 P(+,+|not D)=0.01^2。

最终两种结果应该一致。请给出小数概率或百分比并解释为什么首次阳性和两次阳性结论有很大差别。

提示：这里的独立指的是**给定患病状态的条件独立**，并非说两次阳性事件在总体上无条件独立。

本题答完后进入官方 Exercise 第 2 题（多种 alpha 平滑影响）。

## A/B 测试：为什么胜出概率是 PDF × CDF 的积分？

学生选中先前课程公式：

$$P(\theta_B>\theta_A\mid D)=\int_0^1 f_B(b)F_A(b)\,db,$$

要求“先给我讲一下这个公式我再回答问题”，因此本轮**先暂停官方练习 1**，解释公式本身，不给出练习 1 的数值答案。

### 1. 参数与概率对象

$\theta_A$ 和 $\theta_B$ 是未知的真实点击率；给定现有观测 $D$ 后，A 的参数后验为 Beta$(51,951)$，B 的参数后验为 Beta$(66,936)$。

- $f_B(b)$ 是 $\theta_B$ 的**后验概率密度**（PDF），不是 $P(\theta_B=b\mid D)$。对连续变量，精确取值事件概率为 0，而 $f_B(b)\,db$ 近似表示 B 落在宽为 $db$ 的小区间的概率。
- $F_A(b)=P(\theta_A\le b\mid D)$ 是 A 的后验**累积分布函数**（CDF），描述 A 点击率不超过 $b$ 的概率。
- 特别地，$F_A(0.06)\approx0.9007909938342434$，即在当前 A 后验下 A 点击率不超过 6% 的概率约为 90.08%。

### 2. 固定 B，先看 A 能否低于它

在给定数据 $D$ 后，若两组独立先验且数据各只依赖对应参数，则 $\theta_A,\theta_B$ **后验独立**。当暂时固定 $\theta_B=b$ 时：

$$P(\theta_A<\theta_B\mid\theta_B=b,D)=P(\theta_A<b\mid D)=F_A(b).$$

而 B 落在 $b$ 附近的后验概率近似为 $f_B(b)\,db$。相乘后，对 $b\in[0,1]$ 的所有可能取值积分，得到：

$$P(\theta_B>\theta_A\mid D)=\int_0^1f_B(b)F_A(b)\,db.$$

这是对固定 B 下的胜出概率做关于 B 后验分布的加权平均，亦可由全期望公式理解。

### 3. 双重积分严格推导

在后验独立假设下联合后验密度分解为 $f_{A,B}(a,b)=f_A(a)f_B(b)$，因此

$$\begin{aligned}
P(\theta_B>\theta_A\mid D)
&=\iint_{0\le a<b\le1} f_A(a)f_B(b)\,da\,db\\
&=\int_0^1f_B(b)\left[\int_0^b f_A(a)\,da\right]db\\
&=\int_0^1f_B(b)F_A(b)\,db.
\end{aligned}$$

几何上，$a<b$ 是单位正方形中 $b=a$ 对角线左上方的三角区域。求的是这块区域的**联合后验概率质量**，不能对非均匀分布只量三角形的面积。

### 4. 两个检验例子

如果 A、B 独立服从 Uniform$(0,1)$，则 $f_B(b)=1$、$F_A(b)=b$，从而：

$$P(\theta_B>\theta_A)=\int_0^1 b\,db=1/2.$$

实际 A/B 后验 Beta$(51,951)$ 与 Beta$(66,936)$，用 SciPy beta.pdf、beta.cdf 与 scipy.integrate.quad 数值积分得到

$$P(\theta_B>\theta_A\mid D)\approx0.9244803789025741=92.4480\%.$$

这和此前 1,000,000 次 Monte Carlo 估计约 0.924316（有随机误差）一致。

**本轮不提前给出官方练习 1 两次阳性的结果**；学生理解积分后再继续尝试条件独立的两次阳性贝叶斯更新。

参考数值验证代码见下一单元。

In [ ]:
from scipy.integrate import quad
from scipy.stats import beta

# Posterior for variant A and B:
# A ~ Beta(51,951); B ~ Beta(66,936)
p_b_better, err = quad(
    lambda b: beta.pdf(b, 66, 936) * beta.cdf(b, 51, 951),
    0.0, 1.0, epsabs=1e-12, epsrel=1e-12
)
uniform_check, _ = quad(lambda b: b, 0.0, 1.0)
assert abs(p_b_better - 0.9244803789025741) < 1e-10
assert abs(uniform_check - 0.5) < 1e-12
print('Posterior P(B > A):', p_b_better)
print('Numerical quadrature error estimate:', err)
print('Uniform independent A,B case:', uniform_check)

## PDF × CDF 推导：学生 CDF 理解检查通过

用户原话：

> cdf是计算所有theta_a小于0.06的theta_a累计的概率

**判定：正确。** CDF 的准确数学定义（已给定数据 D）为

$$F_A(b)=P(\theta_A\le b\mid D)=\int_0^b f_A(a)\,da,$$

其中 $\theta_A$ 为定义在 $[0,1]$ 的连续点击率后验随机参数，$f_A$ 是其 PDF。对于 Beta 分布，单点概率为 0，因此严格小于与小于等于的概率相等。

特别地，$F_A(0.06)$ 表示在当前已有观测下，A 的真实点击率不超过 6% 的**后验概率**。它是对密度积分，而不是简单数出一批观测的频数。

联系原公式：给定 $\theta_B=b$ 时，A 小于 B 的条件概率等于 $F_A(b)$（需要 A/B 参数后验独立）。让未知的 B 在 $[0,1]$ 上按 $f_B(b)$ 加权平均，即

$$P(\theta_B>\theta_A\mid D)=\int_0^1F_A(b)f_B(b)\,db.$$

**已通过此概念检查。** 回到上一轮暂停的官方练习 1：在条件独立检测假设下，由两次阳性结果计算真实患病后验。导师尚未提前给出练习数值答案。

## 恢复官方 Exercise 1：两次检测阳性（等待学生）

初始患病率 $P(D)=0.0001$，灵敏度 $P(+\mid D)=0.99$，假阳性率 $P(+\mid\neg D)=0.01$。

给定真实患病与否后，两次检测条件独立，且两次都为阳性。

请使用贝叶斯公式计算 $P(D\mid+,+)$。

方法一：第一次阳性得到 $p_1=P(D\mid+)$，再次把 $p_1$ 当成第二次更新的先验。

方法二：直接利用 $P(+,+\mid D)=0.99^2$ 和 $P(+,+\mid\neg D)=0.01^2$。

先给出结果与思路；教师验收后再进入 Exercise 2。

## 官方练习 1：学生首次解答与独立性辨析（需复算）

学生原始答案：

> 分子=p(+,+|d) \* p(d) = 0.99^2 \* *0.0001 分母=p(+,+)=p(+)*\*p(+) 结果=0.9611687812379852

**评价**：分子正确，但分母错误，因此最终数值错误。这是把两次阳性的**条件独立**误写为**无条件独立**。

假设同一个人做两次检测，在已知其患病/健康状态下两个检测结果相互独立。设 $D$ 为患病、$+_1,+_2$ 为两次阳性。

真阳性与假阳性的联合似然：

$$P(+_1,+_2\mid D)=0.99^2,\qquad P(+_1,+_2\mid\neg D)=0.01^2.$$

所以由全概率公式，联合阳性的边际概率为：

$$\begin{aligned}
P(+_1,+_2)
&=P(+_1,+_2\mid D)P(D)+P(+_1,+_2\mid\neg D)P(\neg D)\\
&=0.99^2\cdot0.0001+0.01^2\cdot0.9999\\
&=0.00009801+0.00009999=0.000198.
\end{aligned}$$

正确后验：

$$\boxed{P(D\mid+_1,+_2)=\frac{0.00009801}{0.000198}=0.495=49.5\%.}$$

学生的答案 $0.9611687812379852$ 刚好对应错误分母 $P(+_1)P(+_2)$。由于每次检验的边际阳性率均为 $P(+)=0.010098$，错误分母为 $0.010098^2=0.000101969604$，从而

$$0.00009801/0.000101969604=0.9611687812379852.$$

关键解释：在已知患病状态时检测误差独立，并不意味着总体上两次阳性无条件独立。两次结果共同依赖隐藏患病状态 $D$，因而边际上相关。第一次阳性提供了患病证据，并改变了第二次阳性的预测概率。

### 序贯更新交叉检验

首次阳性后：

$$p_1=P(D\mid+_1)=\frac{0.99\cdot0.0001}{0.99\cdot0.0001+0.01\cdot0.9999}
=\frac{0.000099}{0.010098}\approx0.00980392157.$$

第二次：

$$p_2=\frac{0.99p_1}{0.99p_1+0.01(1-p_1)}=0.495.$$

数值实测：批量 0.495，序贯 0.49500000000000005。两个结果在浮点精度内一致。

**当前练习状态**：学生首次解答未通过（分子通过，分母有条件独立性错误）。教师已经指出原因并演示正确计算，接下来让学生独立以序贯更新式再写一遍第二次贝叶斯分母，确认理解后进入下一题。

In [ ]:
import math
prior=0.0001
sensitivity=0.99
false_positive_rate=0.01

numerator=(sensitivity**2)*prior
joint_positive=numerator+(false_positive_rate**2)*(1-prior)
posterior_batch=numerator/joint_positive

single_positive=sensitivity*prior+false_positive_rate*(1-prior)
posterior_1=sensitivity*prior/single_positive
posterior_sequential=(sensitivity*posterior_1)/(sensitivity*posterior_1+false_positive_rate*(1-posterior_1))

incorrect_den=single_positive**2
incorrect_result=numerator/incorrect_den

assert math.isclose(posterior_batch,0.495,rel_tol=1e-12)
assert math.isclose(posterior_sequential,posterior_batch,rel_tol=1e-12)
assert math.isclose(incorrect_result,0.9611687812379852,rel_tol=1e-12)
assert not math.isclose(joint_positive,incorrect_den)

print("P(+,+) from total probability:",joint_positive)
print("Incorrect P(+)^2:",incorrect_den)
print("Posterior batch:",posterior_batch)
print("Posterior sequential:",posterior_sequential)
print("Incorrect result reproduced:",incorrect_result)

## 官方练习 1 · 修订检查（等待学生回答）

现在已知首次阳性后：

$$p_1=P(D\mid+_1)=\frac{0.000099}{0.010098}.$$

请使用 $p_1$ 作为第二次贝叶斯更新的先验：

$$P(D\mid+_1,+_2)=\frac{0.99p_1}{\color{blue}{\text{请写出第二次更新分母}}}.$$

把分母展开为“第二次真阳性概率 + 第二次假阳性概率”，并说明为什么不能写成 $P(+)^2$。

这一步确认后进入官方 Exercise 2（平滑参数对分类器的影响）。

## 官方 Exercise 1 · 进一步讲解：为何序贯分母与批量分母不同？

学生虽然能够正确写出批量联合全概率展开：

$$P(+_1,+_2)=P(+_1,+_2\mid D)P(D)+P(+_1,+_2\mid\neg D)P(\neg D)=0.000198,$$

但表示：“我现在对于这个式子的序贯更新还是有一点懵的”。

### 关键区分两个样本空间与分母

批量贝叶斯：

$$P(D\mid+_1,+_2)=\frac{P(+_1,+_2\mid D)P(D)}{P(+_1,+_2)}.$$

批量分母 $P(+_1,+_2)=0.000198$ 是随机抽取一个人，两个测试**一起**阳性的概率。

序贯更新把第一次阳性纳入已知证据，计算：

$$P(D\mid+_1,+_2)=\frac{P(+_2\mid D,+_1)P(D\mid+_1)}{P(+_2\mid+_1)}.$$

序贯第二次的分母 $P(+_2\mid+_1)$ 是从**已知第一次阳性的人群**中，第二次仍然阳性的概率。

因两次检测在给定疾病状态后条件独立：

$$P(+_2\mid D,+_1)=P(+_2\mid D)=0.99,$$
$$P(+_2\mid\neg D,+_1)=P(+_2\mid\neg D)=0.01.$$

记首次阳性后的后验为 $p_1$：

$$p_1=P(D\mid+_1)=\frac{0.99\cdot 0.0001}{0.99\cdot0.0001+0.01\cdot0.9999}=\frac{0.000099}{0.010098}=\frac1{102} \approx0.00980392156862745.$$

第二次阳性用 $p_1$ 作新先验：

$$\boxed{p_2=\frac{0.99p_1}{0.99p_1+0.01(1-p_1)}=0.495.}$$

此时第二次更新的分母为：

$$P(+_2\mid+_1)=0.99p_1+0.01(1-p_1)=\frac{0.000198}{0.010098}\approx0.0196078431372549.$$

两种分母通过条件概率公式联系：

$$P(+_1,+_2)=P(+_1)P(+_2\mid+_1).$$

代入后，序贯后验与批量后验严格一致：

$$\frac{P(+_2\mid D,+_1)P(D\mid+_1)}{P(+_2\mid+_1)}
=\frac{P(+_2\mid D,+_1)P(+_1\mid D)P(D)}{P(+_1,+_2)}
=\frac{P(+_1,+_2\mid D)P(D)}{P(+_1,+_2)}.$$

### 100 万人期望人数示例

假设 100 万人，初始患病 100 人，健康 999900 人：

- 第一次阳性：患病者 $100\times0.99=99$ 人，健康者 $999900\times0.01=9999$ 人，合计 10098 人。
- 第二次仍阳性（在第一次阳性的人群内）：患病者 $99\times0.99=98.01$ 人，健康者 $9999\times0.01=99.99$ 人，合计 198 人。
- 最终患病概率为 $98.01/198=0.495$。

人数是基于概率模型计算的**期望人数**，不是保证出现的小数人数。

特别强调：条件独立不是无条件独立。同一人的两个结果依赖共同隐藏变量 $D$，所以一般不能把 $P(+_1,+_2)$ 写成 $P(+_1)P(+_2)$。

**进度**：本次是对 Exercise 1 的深入解释，未要求学生记忆结果。下一轮只需让学生亲自说明为何序贯分母应写为 $0.99p_1+0.01(1-p_1)$，再进入后续官方练习。

In [ ]:
from math import isclose
p0=0.0001
tp=0.99
fp=0.01
p_plus_1=tp*p0+fp*(1-p0)
p1=tp*p0/p_plus_1
p_plus_2_given_plus_1=tp*p1+fp*(1-p1)
p2_seq=tp*p1/p_plus_2_given_plus_1
p_joint_plus=tp**2*p0+fp**2*(1-p0)
p2_batch=tp**2*p0/p_joint_plus
assert isclose(p_plus_1,0.010098,rel_tol=1e-12)
assert isclose(p1,1/102,rel_tol=1e-12)
assert isclose(p_joint_plus,0.000198,rel_tol=1e-12)
assert isclose(p_plus_2_given_plus_1,p_joint_plus/p_plus_1,rel_tol=1e-12)
assert isclose(p2_seq,0.495,rel_tol=1e-12)
assert isclose(p2_batch,p2_seq,rel_tol=1e-12)
expected_sick=100*tp*tp
expected_healthy=999900*fp*fp
assert isclose(expected_sick/(expected_sick+expected_healthy),p2_seq)
print('First positive marginal:',p_plus_1)
print('First posterior:',p1)
print('Second positive conditional:',p_plus_2_given_plus_1)
print('Joint two-positive marginal:',p_joint_plus)
print('Sequential posterior:',p2_seq)
print('Batch posterior:',p2_batch)

## 官方 Exercise 1 · 最后理解检查

在首次阳性之后，令

$$p_1=P(D\mid+_1)\approx0.00980392.$$

请回答：

**为什么第二次阳性的条件概率**

$$P(+_2\mid+_1)=0.99p_1+0.01(1-p_1),$$

**而不是直接等于 0.99 或等于 $P(+_1)$？**

只需用“第一次阳性人群中仍有患病者和健康者两类人，他们各自第二次阳性的概率不同”这一思路解释即可，不要求重新计算大数值。此题通过后，进入官方 Exercise 2（平滑系数变化）。

## 练习 1 · 补充纠错：为何 P(+2 | +1) 是混合概率，且不妨碍患病后验增长

### 学生新的困惑

学生原话：“其实我是不理解这个 $P(+_2\mid+_1)=0.99p_1+0.01(1-p_1)$ 这个公式怎么来的,还有按照我的理解,连续两次阳性,它是真实患病的概率应该是增加的。”

**反馈**：学生关于第二次阳性提高患病后验的直觉是正确的。困惑来自将下列两个不同条件概率混为一谈：

- $P(+_2\mid+_1)$：已知第一次阳性时，再次阳性的**预测概率**。
- $P(D\mid+_1,+_2)$：已知两次均阳性时，真实患病的**后验概率**。

### 从条件全概率公式推导混合概率

令 $p_1=P(D\mid+_1)=1/102$。患病 $D$ 与健康 $\neg D$ 是互斥且穷尽的划分，因此：

$$\begin{aligned}
P(+_2\mid+_1)
&=P(+_2,D\mid+_1)+P(+_2,\neg D\mid+_1)\\
&=P(+_2\mid D,+_1)P(D\mid+_1)
+P(+_2\mid\neg D,+_1)P(\neg D\mid+_1).
\end{aligned}$$

给定真实疾病状态，两次检测条件独立，所以 $P(+_2\mid D,+_1)=0.99$、$P(+_2\mid\neg D,+_1)=0.01$；带入得：

$$\boxed{P(+_2\mid+_1)=0.99p_1+0.01(1-p_1)=\frac1{51}\approx0.0196078431.}$$

解释：第一次阳性者既包含真正患者（占 $p_1$）也包含健康者（占 $1-p_1$）。前者的第二次阳性概率为 99%，后者为 1%，因此整个第一次阳性人群中的第二次阳性概率是加权平均，不是固定的 99%。

### 两次阳性后患病概率确实上升

利用序贯贝叶斯定理：

$$P(D\mid+_1,+_2)
=\frac{P(+_2\mid D,+_1)P(D\mid+_1)}{P(+_2\mid+_1)}
=\frac{0.99p_1}{0.99p_1+0.01(1-p_1)}=0.495.$$

未检测先验 $0.0001=0.01\%$，第一次阳性后约 $0.980392\%$，两次阳性后 $49.5\%$。因此学生认为“连续两次阳性，患病概率应增加”是完全正确的。

### 自然频率交叉理解

10,000 人中，患病 1 人、健康 9,999 人是期望人口数。第一次阳性的期望人数：患者 $0.99$，健康 $99.99$，合计 $100.98$。这些人第二次也阳性的期望人数：患者 $0.9801$，健康 $0.9999$，合计 $1.98$。

- 第一次阳性之后第二次也阳性的概率：$1.98/100.98\approx1.96078\%$。
- 两次阳性之后真正患病的概率：$0.9801/1.98=49.5\%$。

这两个不同的分母恰好对应两种不同的“已知什么”的条件概率。人数非整数仅表示期望值。

### 接下来的检查

请学生以自己的话区分“已知第一次阳性，再次阳性有多可能”和“已知两次阳性，真正患病有多可能”，暂不进入练习 2，直到该条件概率区别清楚。

## 序贯贝叶斯更新：补全遗漏的条件概率变换

学生指出：此前解释从联合概率切换至条件概率时省略了共同因子的约去；并提出疑问 P(+2,+1) 是否等于 P(+2|+1)。

严格区分：P(+1,+2) = P(+2|+1) P(+1)，两者通常不相等。

从条件概率的定义：

$$P(D\mid+_1,+_2)=\frac{P(D,+_1,+_2)}{P(+_1,+_2)}.$$

分别使用条件乘法公式：

$$P(D,+_1,+_2)=P(D,+_2\mid+_1)P(+_1),$$

$$P(+_1,+_2)=P(+_2\mid+_1)P(+_1).$$

将上述两个公式代入并约去非零的 P(+1)，得到：

$$P(D\mid+_1,+_2)=\frac{P(D,+_2\mid+_1)}{P(+_2\mid+_1)}.$$

再应用条件乘法公式与已知疾病状态下的两次检测条件独立性：

$$P(D,+_2\mid+_1)=P(+_2\mid D,+_1)P(D\mid+_1)=0.99p_1,$$

其中 $p_1=P(D\mid+_1)$。分母按疾病状态用条件全概率公式展开：

$$P(+_2\mid+_1)=0.99p_1+0.01(1-p_1).$$

因此第二次贝叶斯更新：

$$P(D\mid+_1,+_2)=\frac{0.99p_1}{0.99p_1+0.01(1-p_1)}=0.495.$$

数值区分：P(+1,+2)=0.000198，而 P(+2|+1)=0.000198/0.010098 约等于 0.01960784。

教学反思：此前从联合概率写到条件概率时省略了约掉 P(+1) 的步骤，让学生误认为联合概率等于条件概率；本次补充严格推导，不额外催促进入后续练习。

## 练习 1 · 学生辨认序贯分母 P(+2 | +1)

学生原话：

> p(+2|+1),这个就相当于吧这个+1先发生

**判定**：辨认正确。第二次贝叶斯更新的分母是 P(+2 | +1)，其含义是“已经知道/观测到第一次是阳性，在这个条件下再次阳性的预测概率”。先观察并利用 +1 的信息，可以解释序贯更新流程；不过数学上的条件概率仅表示已知事件，不要求两事件一定有时间因果关系。

记第一次阳性之后的患病概率为 p1=P(D|+1)。在给定疾病状态后两次检验条件独立的前提下：

$$P(+_2\mid+_1)=P(+_2\mid D,+_1)P(D\mid+_1)+P(+_2\mid\neg D,+_1)P(\neg D\mid+_1)$$

$$\boxed{P(+_2\mid+_1)=0.99p_1+0.01(1-p_1)}.$$

第二次患病后验是 $P(D\mid+_1,+_2)=0.99p_1/P(+_2\mid+_1)=0.495$。

**联合与条件区别**：$P(+_1,+_2)=P(+_2\mid+_1)P(+_1)$，因此前者为 0.000198，而后者约为 0.01960784。这两个分母属于不同但等价的贝叶斯表达式，不能直接当成相同事件概率。

**小检查**：请学生用一句话说明，P(+2 | +1) 与 P(+1,+2) 分别描述什么样的随机人群事件。理解后继续官方 Exercise 2。

## 官方 Exercise 1：最终理解检查通过

学生原话：

> 因为p(+2|+1) 不等于p(+2|D)

**评价：概念区分正确。** 进一步解释：已知第一次检测阳性并不等于已知受检者患病，前者所对应的人群仍包含患者与健康者；相反，后者完全条件化于患病状态。

因此：

$$P(+_2\mid D)=0.99,$$

$$\begin{aligned}
P(+_2\mid+_1)
&=P(+_2\mid D,+_1)P(D\mid+_1)
 +P(+_2\mid\neg D,+_1)P(\neg D\mid+_1)\\
&=0.99p_1+0.01(1-p_1).
\end{aligned}$$

首次阳性后的患病后验为：

$$p_1=P(D\mid+_1)=1/102.$$

代入得到第二次阳性的条件预测概率：

$$P(+_2\mid+_1)=0.99/102+0.01(101/102)=1/51\approx0.0196078,$$

并不是 0.99。最终的连续两次阳性后患病概率为 0.495，前面批量与序贯计算已经交叉验证。

**本题验收通过**：学生能区分条件化于患病状态与条件化于首次阳性，且理解序贯贝叶斯更新中第二次检测的分母是混合预测概率。后续进入官方练习 2。

## 官方 Exercise 2：平滑系数对条件词概率的影响（等待学生）

继续使用学生已完成并通过验收的 Multinomial NaiveBayes 类。

训练语料：

    train_docs = ["free free win", "meeting schedule", "project meeting", "schedule review"]
    train_labels = ["spam", "ham", "ham", "ham"]

已知 spam 的 token 总数 $N_{\rm spam}=3$；训练词表共有 6 种词：

$$V=\{\mathrm{free},\mathrm{win},\mathrm{meeting},\mathrm{schedule},\mathrm{project},\mathrm{review}\}.$$

对任意加 alpha 平滑：

$$P(w\mid\mathrm{spam})
=\frac{\mathrm{count}(w,\mathrm{spam})+\alpha}{3+6\alpha}.$$

所以：

$$P(\mathrm{free}\mid\mathrm{spam})=\frac{2+\alpha}{3+6\alpha},\qquad
P(\mathrm{meeting}\mid\mathrm{spam})=\frac{\alpha}{3+6\alpha}.$$

请分别使用 $\alpha\in\{0.01,0.1,1,10\}$ 计算两种词概率，并分析随着 $\alpha$ 增大，两者为什么越来越接近。

鼓励学生独立写 Python 循环（或手工计算），不要在回答前预先给出数值表。思考题：当 $\alpha\to\infty$ 时，各词概率趋向什么？对于 $\alpha=0$ 且词在该类别下完全没出现，已有 log likelihood 的 math.log 会发生什么问题？

这一步通过后继续官方 Exercise 3（新增邮件长度特征）、Exercise 4（MAP 计算）。

## 官方 Exercise 2：学生提交、验收与纠错

学生使用自己先前编写的 NaiveBayes 类，独立调整 smoothing in [0.01,0.1,1,10] 并打印两个词的对数条件似然：

    for alpha in [0.01, 0.1, 1, 10]:
        model = NaiveBayes(smoothing=alpha)
        model.train(train_docs,train_labels)
        ll1 = model._log_likelihood("free","spam")
        ll2 = model._log_likelihood("metting","spam")
        print(f"smoothing = {alpha}, likehood(free,spam) = {ll1}, likehood(meeting,spam) = {ll2}")

学生原话：“因为随着alpha的增大,free的次数还有meeting的出现的次数占的分子的比例就越小”。

### 验收与实际测试

对 4 组 $\alpha$，学生打印的 8 个对数条件似然**全部正确**（教师用原 NaiveBayes 实现实际复算）；但是必须指出学生输入第二个单词时拼成了 **metting** 而不是 **meeting**，而输出字符串却写了 meeting。

由于在 spam 类别，正确单词 meeting 和错误单词 metting 的计数都为 0，且 self.word_counts[cls] 使用 defaultdict(int)，两者这一轮计算恰巧一致。然而 metting **不属于训练词汇表 vocab**，是 OOV；在固定词表的多项式概率模型中，应使用真实训练词汇表单词 meeting，否则未登录词被直接按已知词处理会破坏词概率归一化的解释。学生应把代码中的单词拼写修正。

用户打印的结果是 $\log P(w\mid C)$，并非原概率 $P(w\mid C)$。使用 math.exp 恢复后的结果为：

| alpha | P(free given spam) | P(meeting given spam) |
|---:|---:|---:|
| 0.01 | 0.6568627451 | 0.003267973856 |
| 0.1 | 0.5833333333 | 0.02777777778 |
| 1 | 0.3333333333 | 0.1111111111 |
| 10 | 0.1904761905 | 0.1587301587 |

数学模型为：

$$P(\mathrm{free}\mid\mathrm{spam})=\frac{2+\alpha}{3+6\alpha},\qquad
P(\mathrm{meeting}\mid\mathrm{spam})=\frac{\alpha}{3+6\alpha}.$$

概率比值：

$$\frac{P(\mathrm{free}\mid\mathrm{spam})}{P(\mathrm{meeting}\mid\mathrm{spam})}
=1+\frac2\alpha\longrightarrow1.$$

两者均在 $\alpha\to\infty$ 时收敛至训练词表上的均匀概率 $1/6$。因此当平滑越来越强时，平滑伪计数相对于原始 token 计数的影响越来越大，词频差异被削弱；学生的趋势解释方向正确。

当 alpha=0 而某个类别下某词词频为 0 时，当前 log likelihood 会在 math.log(0) 处抛出 ValueError: math domain error。该情况已实际测试。

**状态**：官方 Exercise 2 的计算与趋势理解通过，但保留拼写错误和 OOV 设计风险作为记录；用户应修正 metting 为 meeting。下面提供可复跑的正确代码。

In [ ]:
import math
train_docs=['free free win','meeting schedule','project meeting','schedule review']
train_labels=['spam','ham','ham','ham']
expected=[
    (0.01, 0.6568627450980392, 0.0032679738562091504),
    (0.1, 0.5833333333333334, 0.02777777777777778),
    (1.0, 1/3, 1/9),
    (10.0, 0.1904761904761905, 0.15873015873015878)
]
for alpha,exp_free,exp_meeting in expected:
    model=NaiveBayes(smoothing=alpha)
    model.train(train_docs,train_labels)
    ll_free=model._log_likelihood('free','spam')
    ll_meeting=model._log_likelihood('meeting','spam')  # corrected spelling
    prob_free=math.exp(ll_free)
    prob_meeting=math.exp(ll_meeting)
    assert math.isclose(prob_free,exp_free,rel_tol=1e-12)
    assert math.isclose(prob_meeting,exp_meeting,rel_tol=1e-12)
    print(f'alpha={alpha}, P(free|spam)={prob_free:.9f}, P(meeting|spam)={prob_meeting:.9f}')
    # Mis-spelled OOV currently produces the same count zero in spam, but is not in vocab.
    assert 'meeting' in model.vocab and 'metting' not in model.vocab

model=NaiveBayes(smoothing=0.)
model.train(train_docs,train_labels)
try:
    model._log_likelihood('meeting','spam')
except ValueError as ex:
    assert 'math domain error' in str(ex)
    print('alpha=0 with unseen class word: ValueError as expected')
else:
    raise AssertionError('Expected math.log(0) failure')
print('4 alpha choices * 2 word probabilities verified')

## 官方 Exercise 3：增加邮件长度特征（下一步）

原练习要求：在 NaiveBayes 分类器中增加短/长邮件特征，估计 $P(\mathrm{short}\mid\mathrm{spam})$、$P(\mathrm{short}\mid\mathrm{ham})$，并将其加入预测对数打分。

为了固定实验定义，本轮暂定：

- 对每封邮件使用与训练一致的 lower().split()；
- token 数量 $\le 2$ 定义 short，token 数量 $\ge 3$ 定义 long；
- 使用当前已有训练语料四封邮件；
- 对二元邮件长度特征单独做加一平滑（类别计数分母为该类**邮件数**加 $2$，不是 token 总数加 $2$）。

**先回答两个计算问题**：

1. 当前 spam、ham 的 short、long 各有几封？
2. 平滑后 $P(\mathrm{short}\mid\mathrm{spam})$ 和 $P(\mathrm{short}\mid\mathrm{ham})$ 分别是多少？

提示：对二值长度特征 $L\in\{\mathrm{short},\mathrm{long}\}$，

$$P(L=\mathrm{short}\mid C)=\frac{\#\{\text{C 类 short 邮件}\}+1}{\#\{\text{C 类邮件}\}+2}.$$

回答通过后再让学生独立扩展类的训练统计和 predict()，本轮不提前给出完整实现。

## 官方 Exercise 3：学生长度频率手算正确（2/2）

学生提交：

> 1.spam short= 0 ,spam long = 1, ham short =3 ham long = 0 2.P(short|spam) = 1/3, P(short|ham) = 4/5

**两题全部正确**。

使用四封训练邮件：free free win（spam，3 tokens，long）；meeting schedule（ham，2 tokens，short）；project meeting（ham，2 tokens，short）；schedule review（ham，2 tokens，short）。

类别文档统计：

- spam：short 0，long 1，类别邮件总数 1。
- ham：short 3，long 0，类别邮件总数 3。

对二元长度变量单独加一平滑：

$$P(\mathrm{short}\mid C)=\frac{n_{C,\mathrm{short}}+1}{N_C^{\mathrm{docs}}+2},\quad
P(\mathrm{long}\mid C)=\frac{n_{C,\mathrm{long}}+1}{N_C^{\mathrm{docs}}+2}.$$

从而：

$$P(\mathrm{short}\mid\mathrm{spam})=1/3,\qquad P(\mathrm{long}\mid\mathrm{spam})=2/3,$$

$$P(\mathrm{short}\mid\mathrm{ham})=4/5,\qquad P(\mathrm{long}\mid\mathrm{ham})=1/5.$$

教师实际在 Python 中用 defaultdict(Counter) 遍历训练数据，检查四项类别长度统计和全部平滑概率，计算均匹配。

注意这属于**文档级别**特征，其分母是类别的**文档数+2**，与此前 MultinomialNB 词似然的分母（类别总 token 数 + alpha*词表大小）不同。

方法学提示：长度是输入 token 数量的确定函数，与词频并非在类别条件下严格独立；在原始词似然中再乘长度似然会形成可能重复计权的朴素贝叶斯近似，需要通过评估判断是否有益。

本次仅验收手算与理论，扩展代码尚未提交。

In [ ]:
from collections import defaultdict, Counter
from math import isclose
docs=['free free win','meeting schedule','project meeting','schedule review']
ys=['spam','ham','ham','ham']
length_counts=defaultdict(Counter)
for doc,label in zip(docs,ys):
    tokens=doc.lower().split()
    length='short' if len(tokens)<=2 else 'long'
    length_counts[label][length]+=1
assert length_counts['spam']['short']==0
assert length_counts['spam']['long']==1
assert length_counts['ham']['short']==3
assert length_counts['ham']['long']==0
def smooth(cls,feature):
    return (length_counts[cls][feature]+1)/(sum(length_counts[cls].values())+2)
assert isclose(smooth('spam','short'),1/3)
assert isclose(smooth('spam','long'),2/3)
assert isclose(smooth('ham','short'),4/5)
assert isclose(smooth('ham','long'),1/5)
print('All length feature counts and smoothed probabilities correct.')

## 官方 Exercise 3 · 下一步：修改训练统计（等待学生）

现在请学生独立修改已有 NaiveBayes 类的 __init__ 和 train 两部分，新增**文档级**长度类别统计。

可在 __init__ 新增一个存储结构（可按这个形式定义）：

    self.length_counts = defaultdict(lambda: defaultdict(int))

在 train 中处理每封邮件时，使用和原代码一致的 words.lower().split() 结果，判断是否 short（token 数 <=2），否则 long；然后对 length_counts[label][length] 进行一次计数。

**注意：每封邮件仅更新一次长度统计，不可在每个词的循环中重复计算。**

原有 class_counts、word_counts、class_word_totals、vocab 逻辑保持不变。

训练后应满足：

    assert model.length_counts["spam"]["short"] == 0
    assert model.length_counts["spam"]["long"] == 1
    assert model.length_counts["ham"]["short"] == 3
    assert model.length_counts["ham"]["long"] == 0

本轮请学生先提交自己写的 __init__ 和 train，不直接给出完整答案。验收通过后再实现 _log_length_likelihood() 与合并进入 predict()。

提示：长度是词序列的确定函数，因此长度特征和词频特征严格来说并不条件独立，这是一种朴素贝叶斯的额外建模近似。

## 官方 Exercise 3：学生独立扩展长度训练统计，21/21 测试通过

学生在 __init__ 中新增

    self.length_counts = defaultdict(lambda: defaultdict(int))

并在 train() 的每封邮件开始位置对 token 数量做一次 short/long 分类：

    if len(words.split()) > 2:
        self.length_counts[label]["long"] += 1
    else:
        self.length_counts[label]["short"] += 1

这一段正确放在逐词 for 循环之外，因此长度计数每封邮件仅增加一次。其余 class_counts、word_counts、class_word_totals、vocab 以及原先 predict 的计算逻辑未被破坏。

**实际在 Python 中运行学生完整类进行 21 项测试，全部通过。** 测试范围：4 项 short/long 计数，2 项文档类别数，3 项样本词频，2 项 token 总计数，词表大小，4 项预测（含大小写、重复空白与换行），混合空白与空邮件长度计数，以及对全部类别检查长度计数和 = 该类别文档数。

当前四封训练邮件：

| 类别 | short | long |
|---|---:|---:|
| spam | 0 | 1 |
| ham | 3 | 0 |

类别级不变量：

$$\mathrm{length\_counts}[C][\mathrm{short}]+\mathrm{length\_counts}[C][\mathrm{long}]=\mathrm{class\_counts}[C].$$

细节改进：学生写了 len(words.split()) 和 words.lower().split() 两次分词，尽管这四封纯英文邮件下结果一致，也能处理典型空白字符，但可以先统一生成 tokens=words.lower().split()，后续同时用于长度判断和逐 token 计数，以避免重复工作、统一预处理定义。

注意现有 predict() 尚未包含长度的贡献，这正是下一子任务。正式 Exercise 3 还没有全部完成。

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()
        self.length_counts = defaultdict(lambda: defaultdict(int))

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            if len(words.split()) > 2:
                self.length_counts[label]["long"] += 1
            else:
                self.length_counts[label]["short"] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        # 返回 log P(word | cls)
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

    def _log_prior(self, cls):
        # 返回 log P(cls)
        return math.log(self.class_counts[cls])  - math.log(sum(self.class_counts.values()))

    def predict(self, document):
        # 将邮件转换为小写并分词
        # 遍历 self.class_counts 中的每个类别
        # 初始分数为 self._log_prior(cls)
        # 对每个词累加 self._log_likelihood(word, cls)
        # 返回得分最高的类别
        words = document.lower().split()
        pre = {}
        for cls in self.class_counts:
            pre[cls] = self._log_prior(cls) + sum(self._log_likelihood(word,cls) for word in words)
        return max(pre,key=pre.get)

In [ ]:
from math import isclose
m=NaiveBayes()
m.train(['free free win','meeting schedule','project meeting','schedule review'],['spam','ham','ham','ham'])
checks={
    'spam short 0':m.length_counts['spam']['short']==0,
    'spam long 1':m.length_counts['spam']['long']==1,
    'ham short 3':m.length_counts['ham']['short']==3,
    'ham long 0':m.length_counts['ham']['long']==0,
    'spam class 1':m.class_counts['spam']==1,
    'ham class 3':m.class_counts['ham']==3,
    'spam free 2':m.word_counts['spam']['free']==2,
    'spam win 1':m.word_counts['spam']['win']==1,
    'ham schedule 2':m.word_counts['ham']['schedule']==2,
    'spam token count 3':m.class_word_totals['spam']==3,
    'ham token count 6':m.class_word_totals['ham']==6,
    'vocab size 6':len(m.vocab)==6,
    'free win spam':m.predict('free win')=='spam',
    'meeting schedule ham':m.predict('meeting schedule')=='ham',
    'upper case':m.predict('FREE WIN')=='spam',
    'arbitrary whitespace':m.predict('  free   \nWIN ')=='spam'
}
m2=NaiveBayes()
m2.train(['  ONE\t TWO\nTHREE ','one \tTWO', ''],['spam','ham','ham'])
checks.update({
    'long split with whitespace':m2.length_counts['spam']['long']==1,
    'short split with whitespace':m2.length_counts['ham']['short']==2,
    'length once per spam doc':sum(m.length_counts['spam'].values())==m.class_counts['spam'],
    'length once per ham doc':sum(m.length_counts['ham'].values())==m.class_counts['ham'],
    'length once per mixed doc':all(sum(m2.length_counts[c].values())==m2.class_counts[c] for c in m2.class_counts)
})
assert len(checks)==21 and all(checks.values()),checks
print(f'Length feature train extension: {sum(checks.values())}/{len(checks)} passed')

## 官方 Exercise 3 · 第三步：实现长度特征的对数似然

已经完成长度统计的训练部分（21/21 项测试通过）。现在请学生独立编写一个方法，仍沿用加一平滑，并让长度特征始终只包含 short 和 long 两种取值。

待实现：

    def _log_length_likelihood(self, length, cls):
        # 返回 log P(length | cls)
        pass

因为 length 是二元的**邮件级特征**，正确公式为：

$$\boxed{\log P(L\mid C)=\log\left(\frac{\mathrm{length\_counts}[C][L]+1}{\mathrm{class\_counts}[C]+2}\right)}.$$

不要把现有 self.class_word_totals 用作分母，因为这里统计的是每类**邮件**的长度，而不是每类总 token 数。

用之前同一组训练数据，验收应当满足：

    assert math.isclose(model._log_length_likelihood('short','spam'), math.log(1/3))
    assert math.isclose(model._log_length_likelihood('long','spam'), math.log(2/3))
    assert math.isclose(model._log_length_likelihood('short','ham'), math.log(4/5))
    assert math.isclose(model._log_length_likelihood('long','ham'), math.log(1/5))

本轮只提交这个新方法，不要求提前修改 predict。通过后再把长度的 log likelihood 加到每个类别的评分：

$$\operatorname{score}(C;d)=\log P(C)+\sum_w\log P(w\mid C)+\log P(L(d)\mid C).$$

解释上的限定：长度可从词序列计算出来，与词频并非严格条件独立，因此附加长度分数会产生可能的重复证据加权；作为练习先完成实现，最终是否改善预测要在独立评估集上检验。

## 官方 Exercise 3 · 第三步：学生长度对数似然方法验收

学生原始提交：

    def _log_length_likelihood(self, length, cls):
            # 返回 log P(length | cls)
            length_counts = math.log(self.length_counts[cls][length] + self.smoothing)
            class_counts = math.log(self.class_counts[cls] + self.smoothing * 2)
            return length_counts - class_counts

**代码正确。** 对正的 alpha=self.smoothing，实际计算：

$$\log P(L\mid C)=\log(n_{C,L}+\alpha)-\log(N_C^{\rm docs}+2\alpha).$$

其中长度取值的类别数为 2，分母必须使用该类的**邮件总数**，不能使用 token 总数。

### 独立实际验收：36/36 通过

教师在 Python 中把学生这轮方法加到他此前通过测试的完整 NaiveBayes 类，使用四封邮件训练，分别对 alpha = [0.01,0.1,0.5,1,2,10] 运行：

- 对每个 alpha，两种类别 spam/ham 和两种长度 short/long 的概率与数学公式比较，共 6×2×2=24 项；
- 对每个 alpha 和类别，计算 exp(log probability(short))+exp(log probability(long))，验证等于 1，共 6×2=12 项。

结果 36/36 全部通过。alpha=1 的四个概率为 spam short 1/3、spam long 2/3、ham short 4/5、ham long 1/5。

原方法的两个局部变量 length_counts 和 class_counts 已经过 math.log，实际上是对数分子/分母，建议改名 log_numerator 和 log_denominator，但保留学生原始代码，不自动改写。

**接口边界**：应限制传入的 length 为 short/long，且 alpha > 0。当前 defaultdict(int) 访问未知长度会创建一个新键，固定二分类分母却仍加 2 alpha，无法作为第三个长度类别的归一化概率；当前练习仅针对两种有效长度。未训练类别或训练集空、平滑非正也未覆盖。

**课程进度**：长度特征的统计方法、数学概率和对数似然均已通过。最后一步让学生把 log length likelihood 加入 predict，再根据有无特征对比模型表现。

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()
        self.length_counts = defaultdict(lambda: defaultdict(int))

    def train(self, documents, labels):
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            if len(words.split()) > 2:
                self.length_counts[label]["long"] += 1
            else:
                self.length_counts[label]["short"] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

    def _log_prior(self, cls):
        return math.log(self.class_counts[cls]) - math.log(sum(self.class_counts.values()))

    def _log_length_likelihood(self, length, cls):
        # 返回 log P(length | cls)
        length_counts = math.log(self.length_counts[cls][length] + self.smoothing)
        class_counts = math.log(self.class_counts[cls] + self.smoothing * 2)
        return length_counts - class_counts

    def predict(self, document):
        words = document.lower().split()
        pre = {}
        for cls in self.class_counts:
            pre[cls] = self._log_prior(cls) + sum(self._log_likelihood(word,cls) for word in words)
        return max(pre,key=pre.get)

In [ ]:
import math

test_docs=["free free win","meeting schedule","project meeting","schedule review"]
test_labels=["spam","ham","ham","ham"]
results=[]
for alpha in (0.01,0.1,0.5,1.,2.,10.):
    model=NaiveBayes(smoothing=alpha)
    model.train(test_docs,test_labels)
    for cls in ("spam","ham"):
        for length in ("short","long"):
            observed=model._log_length_likelihood(length,cls)
            expected=math.log((model.length_counts[cls][length]+alpha)/(model.class_counts[cls]+2*alpha))
            results.append(math.isclose(observed,expected,rel_tol=1e-12,abs_tol=1e-12))
        total=sum(math.exp(model._log_length_likelihood(length,cls)) for length in ("short","long"))
        results.append(math.isclose(total,1.,rel_tol=1e-12,abs_tol=1e-12))
assert len(results)==36 and all(results)
print(f"Student length log likelihood: {sum(results)}/{len(results)} tests passed.")
m=NaiveBayes()
m.train(test_docs,test_labels)
print({(cls,L):round(math.exp(m._log_length_likelihood(L,cls)),9) for cls in ("spam","ham") for L in ("short","long")})

## 官方 Exercise 3 · 最后一步：把长度似然加入 predict()

目前学生的 predict() 仍然只用词频与先验。请学生**独立修改 predict() 一个方法**，其余所有已通过验收的方法保持不变。

定义与训练阶段一致的长度标签：

    length = "short" if len(words) <= 2 else "long"

新的分类得分：

$$\operatorname{score}(C;d)
=\log P(C)+\sum_{w\in d}\log P(w\mid C)
+\log P(L(d)\mid C).$$

请在每类得分上增加一次长度特征的对数似然，注意每封邮件只加一次，而不是每遇到一个 token 就加一次。仍使用 max(pre, key=pre.get) 返回最高得分类别。

**验证数据：**

    model = NaiveBayes()
    model.train(
      ["free free win","meeting schedule","project meeting","schedule review"],
      ["spam","ham","ham","ham"]
    )

    assert model.predict("free win") == "spam"
    assert model.predict("meeting schedule") == "ham"

思考题：加入长度特征后，单词 free 单独构成一封 short 邮件时，是否必然与原始无长度特征的预测相同？请用先验、词似然和长度似然三个组成部分解释变化，而不是凭印象判断。

请提交修改后的 predict() 与你的预期结果。本题完成后检验新增长度特征是否真能改善独立测试集的表现，并进入官方 Exercise 4（MAP by hand）。

## 官方 Exercise 3：学生长度特征 predict 最终验收（65/65）

### 学生提交的修改

学生在 predict 内用 words=document.lower().split() 规范化后，定义 length="short" if len(words)<=2 else "long"，然后对每个类别计算

$$s(C;d)=\log P(C)+\sum_{w\in d}\log P(w\mid C)+\log P(L(d)\mid C).$$

长度项在每个类别得分里**只加一次**，实现与题目完全一致；使用 max(pre,key=pre.get) 返回得分类别名称。下一个代码单元保存包含学生原始 predict 的整个 NaiveBayes 类，而不是仅给出改写后的答案。

### 在 Python 中实际执行 65/65 检查

- 15 个测试文本的分类预测，包含空消息、重复 token、大小写与混合空白；
- 4 个长度似然值与理论分数比较、2 个类别各自长度概率归一化；
- 4 种文本×2 类共 8 项，确保新版得分与旧版得分的差恰好是**一个**长度 log likelihood；
- alpha 属于 0.01、0.1、0.5、1、2、10，3 种文本、2 种类别，共 36 项验证直接相乘的概率与 log 累加后的概率相等。

合计 $15+4+2+8+36=65$，全部通过。测试代码可在下个代码单元自行重跑。

### 与原版不同的决策

当前训练语料：spam 有 1 封、长度 long、free 词频 2；ham 有 3 封、长度均为 short。

- free：原版仅词频，$S_{\rm spam}=1/12>S_{\rm ham}=1/16$，预测 spam；加长度项后，$S'_{\rm spam}=(1/12)(1/3)=1/36$，$S'_{\rm ham}=(1/16)(4/5)=1/20$，预测 **ham**。
- free win：原版 spam 与 ham 未归一化概率为 $1/54$ 和 $1/192$；加入 short 后乘 $1/3$ 和 $4/5$，得到 $1/162$ 与 $1/240$，仍预测 **spam**。
- meeting schedule：原版、新版都预测 ham。

**解释：长度似然能够改变模型决策，但这并不证明性能改善。** 长度可由词序列直接确定，不满足给定类别后的严格独立性假设；加权时可能重复利用同一文本中的信息。只有在独立测试集验证后才能讨论收益。

### 当前实现的工程边界

训练阶段 len(words.split()) 与 word.lower().split() 各做一次分词；对于当前按空白计数的英语语料功能正确，但可统一为 tokens=words.lower().split() 避免冗余。长度只接受 short/long。未登录词 OOV、空训练集、非法 alpha 尚未覆盖，本次验收不宣称完整工业级健壮性。

**课程状态：官方 Exercise 3 完成。下一题是官方 Exercise 4：用 Beta(2,2) 先验、7 正 3 反，重新独立推导 MAP 与 MLE 的比较。**

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()
        self.length_counts = defaultdict(lambda: defaultdict(int))

    def train(self, documents, labels):
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            if len(words.split()) > 2:
                self.length_counts[label]["long"] += 1
            else:
                self.length_counts[label]["short"] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_likelihood(self, word, cls):
        return math.log(self.word_counts[cls][word] + self.smoothing) - math.log(self.class_word_totals[cls] + self.smoothing * len(self.vocab))

    def _log_prior(self, cls):
        return math.log(self.class_counts[cls]) - math.log(sum(self.class_counts.values()))

    def _log_length_likelihood(self, length, cls):
        # 返回 log P(length | cls)
        length_counts = math.log(self.length_counts[cls][length] + self.smoothing)
        class_counts = math.log(self.class_counts[cls] + self.smoothing * 2)
        return length_counts - class_counts

    def predict(self, document):
        # 将邮件转换为小写并分词
        # 遍历 self.class_counts 中的每个类别
        # 初始分数为 self._log_prior(cls)
        # 对每个词累加 self._log_likelihood(word, cls)
        # 返回得分最高的类别
        words = document.lower().split()
        length = "short" if len(words) <= 2 else "long"
        pre = {}
        for cls in self.class_counts:
            pre[cls] = self._log_prior(cls) + sum(self._log_likelihood(word,cls) for word in words) + self._log_length_likelihood(length,cls)
        return max(pre,key=pre.get)

In [ ]:
import math
docs=['free free win','meeting schedule','project meeting','schedule review']
labels=['spam','ham','ham','ham']
m=NaiveBayes()
m.train(docs,labels)
checks={}
expected={
 'free win':'spam', 'meeting schedule':'ham', 'FREE WIN':'spam',
 '  Free  win  ':'spam', 'free':'ham', 'win':'ham',
 'free free win':'spam', 'project meeting':'ham',
 'schedule review':'ham','meeting':'ham', 'schedule':'ham',
 '':'ham','free meeting':'ham','free free':'spam','free free free':'spam',
}
for text,y in expected.items():
    checks[f'predict {text!r}']=m.predict(text)==y
for cls in ('spam','ham'):
    for length in ('short','long'):
        value=math.exp(m._log_length_likelihood(length,cls))
        target={'spam':{'short':1/3,'long':2/3},
                'ham':{'short':4/5,'long':1/5}}[cls][length]
        checks[f'length probability {cls} {length}']=math.isclose(value,target,rel_tol=1e-12)
    norm=sum(math.exp(m._log_length_likelihood(L,cls)) for L in ('short','long'))
    checks[f'length normalization {cls}']=math.isclose(norm,1.,rel_tol=1e-12)

def scores(model,doc,with_length):
    words=doc.lower().split()
    length='short' if len(words)<=2 else 'long'
    return {cls: model._log_prior(cls) + sum(model._log_likelihood(w,cls) for w in words)
                 + (model._log_length_likelihood(length,cls) if with_length else 0)
            for cls in model.class_counts}
for doc in ('free','free win','free free win','meeting schedule'):
    before=scores(m,doc,False)
    after=scores(m,doc,True)
    length='short' if len(doc.lower().split())<=2 else 'long'
    for cls in m.class_counts:
        checks[f'length added once {doc} {cls}']=math.isclose(
            after[cls]-before[cls],m._log_length_likelihood(length,cls),
            rel_tol=1e-12,abs_tol=1e-12)
for alpha in (0.01,0.1,0.5,1.,2.,10.):
    x=NaiveBayes(smoothing=alpha)
    x.train(docs,labels)
    for doc in ('free','free win','meeting schedule'):
        words=doc.lower().split()
        length='short' if len(words)<=2 else 'long'
        for cls in x.class_counts:
            exact=(x.class_counts[cls]/sum(x.class_counts.values()))
            exact*=math.prod((x.word_counts[cls][w]+alpha)/
                             (x.class_word_totals[cls]+alpha*len(x.vocab))
                             for w in words)
            exact*=(x.length_counts[cls][length]+alpha)/(x.class_counts[cls]+2*alpha)
            checks[f'direct probability alpha={alpha} {doc} {cls}']=math.isclose(
                math.exp(scores(x,doc,True)[cls]),exact,
                rel_tol=1e-12,abs_tol=1e-12)
assert len(checks)==65 and all(checks.values()), [k for k,v in checks.items() if not v]
print(f'Length feature predict: {sum(checks.values())}/{len(checks)} tests passed.')
for doc in ('free','free win','meeting schedule'):
    old=scores(m,doc,False)
    new=scores(m,doc,True)
    print(doc, 'old:',max(old,key=old.get),'new:',max(new,key=new.get))

## 官方 Exercise 4：MAP by hand（等待学生）

第 5 讲我们曾用这个例子解释 MLE、MAP、后验均值的区别。现在作为官方练习，请你重新**独立计算并解释**。

已知硬币正面概率 $\theta$ 未知，10 次独立抛掷中有 7 次正面、3 次反面；使用先验 Beta$(2,2)$。

1. 写出似然函数（允许省略与 $\theta$ 无关的二项式组合常数）。
2. 写出 $\theta$ 的后验 Beta 分布及其参数。
3. 计算 $\hat\theta_{\rm MAP}$ 和 $\hat\theta_{\rm MLE}$，比较哪一个更靠近先验中心 $0.5$，并解释原因。

给定可用于验证的公式（后验 Beta$(a',b')$、且 $a',b'>1$）：

$$\theta_{\rm MAP}=\frac{a'-1}{a'+b'-2}.$$

注意：不要把 MAP 与后验均值 $\mathbb E[\theta\mid D]=a'/(a'+b')$ 混淆。

完成本题后进行本课完整小结、原始 3 题课后测试及学习记录总结。

## 官方 Exercise 4：学生首次提交与讲评（待简短修正）

学生原始答案：

> 1,L(theta) = theta^7 * (1-theta)^3 2.theta~Beta(9,5),3.theta_mle = 0.7 theta_Map = 0.5

**验收：四个判断中 3 个正确，MAP 数值错误。**

1. 似然函数正确：$L(\theta)\propto\theta^7(1-\theta)^3$（省略常数不影响极大值）。
2. 后验参数正确：$\theta\mid D\sim\mathrm{Beta}(2+7,2+3)=\mathrm{Beta}(9,5)$。写条件化于 $D$ 比单独写 $\theta\sim$ 更严谨。
3. MLE 正确：$\hat\theta_{\rm MLE}=7/10=0.7$。
4. MAP 错误：0.5 是**先验** Beta$(2,2)$ 的众数，不是**后验** Beta$(9,5)$ 的众数。

正确的后验 MAP：

$$\hat\theta_{\rm MAP}=\frac{9-1}{9+5-2}=\frac8{12}=\frac23\approx0.666667.$$

完整关系是：

$$\mathrm{prior\ mode}=0.5 < \hat\theta_{\rm MAP}=2/3 < \hat\theta_{\rm MLE}=0.7.$$

更严格的推导：后验密度核 $\theta^{8}(1-\theta)^4$，求导其对数 $\ell_{\rm post}(\theta)=8\log\theta+4\log(1-\theta)+\mathrm{const}$：

$$\ell'_{\rm post}(\theta)=8/\theta-4/(1-\theta)=0 \implies \theta=2/3.$$

后验均值 $9/14$ 则又是不同的汇总量；题目需要后验众数 MAP。

**真实 Python 复核**：使用 Python math.isclose 检查 MLE=0.7、后验参数 (9,5)、MAP=2/3、先验众数=0.5、后验均值=9/14，以及 MAP 介于先验众数和 MLE 之间等 7 项断言，7/7 通过。学生答案仍应独立订正，暂不标记本题完成。

**下一步**：让学生重新给出 MAP 的准确分数，并用一句话区别“先验众数”和“后验众数”。确认后进入官方 3 题 post-quiz。

In [ ]:
from math import isclose
s,f=7,3
a,b=2,2
p_a,p_b=a+s,b+f
mle=s/(s+f)
prior_mode=(a-1)/(a+b-2)
map_mode=(p_a-1)/(p_a+p_b-2)
posterior_mean=p_a/(p_a+p_b)
checks={
    'MLE': isclose(mle,0.7),
    'posterior a': p_a==9,
    'posterior b': p_b==5,
    'MAP': isclose(map_mode,2/3),
    'prior mode': isclose(prior_mode,0.5),
    'prior < MAP < MLE': prior_mode<map_mode<mle,
    'posterior mean': isclose(posterior_mean,9/14),
}
assert len(checks)==7 and all(checks.values()),checks
print(f'Analytical checks: {sum(checks.values())}/{len(checks)} passed')
print('prior mode:',prior_mode,'MAP:',map_mode,'MLE:',mle,'posterior mean:',posterior_mean)

## 官方 Exercise 4 · 最后订正（等待学生）

学生已经正确给出似然 $\theta^7(1-\theta)^3$、后验 Beta$(9,5)$、MLE $0.7$，但把先验众数 $0.5$ 当成了后验 MAP。

请用 Beta 众数公式重新计算：

$$\hat\theta_{\rm MAP}=\frac{a'-1}{a'+b'-2},\quad (a',b')=(9,5).$$

**需要学生只回答两项：**

1. $\hat\theta_{\rm MAP}$ 的准确分数是多少？
2. 为什么它比 MLE 的 0.7 更接近 0.5，但不直接等于 0.5？

订正通过后，进入原课程正式 3 题后测，并在后测全部完成后汇总更新学习进度。

## 官方 Exercise 4：学生订正尝试（概念需继续修正）

学生原话：

> p(theta)是0.5,Map= mle*先验 所以介于 0.5和0.7之间

**学生已经理解方向**：先验偏好 0.5、观测数据的 MLE 为 0.7；两者共同决定 MAP。但对参数、概率密度、优化算子仍有明显混淆，不能直接算已全部通过。

严格区分：

1. 对于 Beta$(2,2)$ 先验，**先验众数**是 $\theta_{\mathrm{prior\ mode}}=0.5$，**先验概率密度函数**为 $p(\theta)=6\theta(1-\theta)$（$0<\theta<1$）。不是 $p(\theta)=0.5$。例如 $p(0.5)=1.5$，这是密度数值而不是离散概率。
2. $\hat\theta_{\mathrm{MLE}}=0.7$ 是使似然最大的**参数值**。似然函数是 $p(D\mid\theta)\propto\theta^7(1-\theta)^3$；MAP 不是把 MLE 数值乘以先验众数或“先验”。
3. MAP 正确优化目标是：

$$\hat\theta_{\mathrm{MAP}}=\arg\max_\theta p(D\mid\theta)p(\theta).$$

结合 Beta$(2,2)$ 先验：

$$p(D\mid\theta)p(\theta)\propto\theta^7(1-\theta)^3\cdot\theta(1-\theta)=\theta^8(1-\theta)^4.$$

对数导数条件 $8/\theta-4/(1-\theta)=0$ 得 $\hat\theta_{\mathrm{MAP}}=8/12=2/3$。

MAP $2/3$ 在本例介于先验众数 0.5 与 MLE 0.7 之间，但“介于二者之间”不能不加条件地当作任意模型的普适定理。

**下一步简短检查（先不开始 post-quiz）**：让学生回答 MAP 到底是“MLE 数值乘先验众数”，还是“找一个 $\theta$，让似然函数与先验密度乘积达到最大”？希望学生使用 argmax 和函数概念自行说明区别。通过后再进入原教程的 3 题后测。

## 众数、argmax：学生理解检查

> 什么是众数呢 选择B,argmax就是返回使得似然函数与先验密度的乘积的theta

判断正确。众数是离散数据中出现最频繁的值；连续分布中是概率密度函数最大的位置。

先验 Beta(2,2) 众数为 0.5，观察到 7 次正面、3 次反面，后验 Beta(9,5) 的众数（MAP）为 2/3。

argmax 返回使目标函数最大的参数，而 max 返回最大的目标函数值。MAP 最大化的是似然与先验密度的乘积。

**官方练习 4 概念订正通过，进入正式后测 1/3。**

## 正式后测第 1/3 题

朴素贝叶斯分类器中的拉普拉斯平滑是什么？为什么需要它？

A. 降低特征空间的维度。
B. 为特征计数增加平滑项，避免未出现的词出现零概率使似然连乘变为零。
C. 平滑决策边界。
D. 将特征标准化为零均值单位方差。

请先选择 A/B/C/D 并解释；此处不公布答案。